# Kaggle run: run C = fine-tuned e5 candidates + full-data stage 5 (runs in the background)
**Setup (once, right-hand panel):**
1. **Add Input -> Your Work -> Datasets**: add `mlc26-core`, `mlc26-emb`, `mlc26-v3`, `mlc26-ftmodel`.
2. **Session options: Accelerator GPU T4 x2, Internet On.**
3. Click **Save Version -> Save & Run All (Commit)**. It runs in the background (browser and laptop can be off), up to 12 h.

**Outputs** (Output tab of the saved version; the log is `log_kaggle.txt`):
- `v3s5ft_fr90/matching_results.tsv`: submit first (full-data stage 5, t 0.80 / France 0.90), ready after ~5 h
- `blend56ft_fr90/matching_results.tsv`: stage 5 + a second model averaged (usually a little better), ~6.5 h
- `v3s5ft/`, `blend56ft/`: the same at the OOF-best threshold; `candidates/candidate_pairs.tsv` for the final zip.
The OOF score of each is in the log: `OOF best t=... F0.5=...`.

In [ ]:
import subprocess, os
print(subprocess.run("pip -q install rapidfuzz anyascii 2>&1 | tail -1; nproc; free -g | head -2; nvidia-smi -L; find /kaggle/input -maxdepth 3 -type d | head -20",
                     shell=True, capture_output=True, text=True).stdout)
os.makedirs("/kaggle/working/src", exist_ok=True)

In [ ]:
%%writefile /kaggle/working/src/common.py
"""Paths + shared helpers. Every stage reads/writes parquet under WORK so re-runs skip finished stages."""
import os
import time
from pathlib import Path

import polars as pl

HERE = Path(__file__).resolve().parent
DATA = Path(os.environ.get("BER_DATA", HERE.parents[2] / "6ab10eb3b23ba_student_resource" / "student_resource" / "dataset"))
WORK = Path(os.environ.get("BER_WORK", HERE.parents[2] / "work"))
OUT = Path(os.environ.get("BER_OUT", HERE.parents[2] / "output"))
WORK.mkdir(parents=True, exist_ok=True)
SPLITS = ("train", "test")
# Version tag for everything a model run writes (p*, xgb_*, comp*, ce*, val_*). "" = v2 names (unchanged);
# "_v3" = blocking v3. FEAT = which feature files the models read (feat2 = v2, feat3 = v3 with kNN pairs).
TAG = os.environ.get("BER_TAG", "")
FEAT = os.environ.get("BER_FEAT", "feat2")


def encode_ids(col):
    """'S2-681193310' -> 2_681193310 (int64). Source digit * 1e9 + number."""
    c = pl.col(col)
    return (c.str.slice(1, 1).cast(pl.Int64) * 1_000_000_000 + c.str.slice(3).cast(pl.Int64))


def decode_ids(col):
    c = pl.col(col)
    return pl.concat_str([pl.lit("S"), (c // 1_000_000_000).cast(pl.Utf8), pl.lit("-"),
                          (c % 1_000_000_000).cast(pl.Utf8)])


def done(path):
    return Path(path).exists()


class timer:
    def __init__(self, msg):
        self.msg = msg

    def __enter__(self):
        self.t = time.time()
        print(f"> {self.msg}", flush=True)

    def __exit__(self, *a):
        print(f"  {self.msg}: {time.time() - self.t:.0f}s", flush=True)


def f05_scores(s1_all, pred, truth):
    """s1_all: [s1] every S1 scored (singletons count). pred/truth: [s1, id] pairs.
    Per S1: F0.5 = 1.25*tp / (0.25*n_true + n_pred); both empty -> 1."""
    tp =pred.join(truth, on=["s1", "id"], how="inner").group_by("s1").len("tp")
    npred = pred.group_by("s1").len("np")
    ntrue = truth.group_by("s1").len("nt")
    df = (s1_all.select("s1").join(tp, on="s1", how="left").join(npred, on="s1", how="left")
          .join(ntrue, on="s1", how="left").fill_null(0))
    f = pl.when((pl.col("np") == 0) & (pl.col("nt") == 0)).then(1.0).otherwise(
        1.25 * pl.col("tp") / (0.25 * pl.col("nt") + pl.col("np")).clip(lower_bound=1e-9))
    return df.with_columns(f.alias("f"))


def f05(s1_all, pred, truth):
    return f05_scores(s1_all, pred, truth)["f"].mean()

In [ ]:
%%writefile /kaggle/working/src/s0_load.py
"""Stage 0: TSV -> parquet, once. Records: [id:int64, src:int8, name, addr, country]. GT: [s1, id] pairs."""
import polars as pl

from common import DATA, SPLITS, WORK, done, encode_ids, timer


def read_tsv(path):
    # quote_char=None: fields contain raw quotes; tabs are the only delimiter
    return pl.scan_csv(path, separator="\t", quote_char=None, infer_schema=False)


def main():
    for split in SPLITS:
        out = WORK / f"{split}_records.parquet"
        if done(out):
            continue
        with timer(f"load {split}"):
            lf = pl.concat([read_tsv(DATA / split / f"{split}_source{k}.tsv") for k in (1, 2, 3)])
            df = lf.select(
                encode_ids("entity_id").alias("id"),
                pl.col("entity_id").str.slice(1, 1).cast(pl.Int8).alias("src"),
                pl.col("business_name").fill_null("").alias("name"),
                pl.col("business_address").fill_null("").alias("addr"),
                pl.col("country").fill_null("").cast(pl.Categorical).alias("country"),
            ).collect()
            assert df["id"].is_unique().all(), "duplicate ids"
            print(split, df.group_by("src").len().sort("src").rows())
            df.write_parquet(out)
            del df

    out = WORK / "train_gt.parquet"
    if not done(out):
        with timer("load gt"):
            gt = (read_tsv(DATA / "train" / "train_ground_truth.tsv")
                  .select(encode_ids("source1_entity_id").alias("s1"),
                          pl.col("matched_entity_ids").fill_null("").str.split(",").alias("id"))
                  .explode("id").filter(pl.col("id") != "")
                  .select("s1", encode_ids("id").alias("id")).collect())
            print("gt pairs", gt.height)
            gt.write_parquet(out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/src/s1_normalize.py
"""Stage 1: normalize names/addresses (country-agnostic rules; France unseen in train).
Native scripts are transliterated to ASCII (anyascii) first, so Devanagari/Odia/Gujarati names compare to English.
Out per split: id, src, country, name_n, core, nosp, skel, addr_n, nums(list), atoks(list), stoks(list)."""
import re

import polars as pl
from anyascii import anyascii

from common import SPLITS, WORK, done, timer

# ponytail: hand-written maps, extend from error analysis
LEGAL_MAP = {"private": "pvt", "pvt": "pvt", "limited": "ltd", "corporation": "corp", "company": "co",
             "incorporated": "inc", "compagnie": "cie", "praivet": "pvt", "praivett": "pvt", "pra": "pvt",
             "prai": "pvt", "li": "ltd", "limitedd": "ltd", "limited": "ltd", "limitedda": "ltd", "elelpi": "llp",
             "enterprises": "ent", "enterprise": "ent", "services": "svc", "service": "svc",
             "industries": "ind", "industry": "ind", "technologies": "tech", "technology": "tech",
             "international": "intl", "associates": "assoc", "brothers": "bros"}
LEGAL = {"pvt", "ltd", "llc", "llp", "inc", "corp", "co", "cie", "sarl", "sas", "sasu", "eurl", "sa", "sci",
         "ei", "pllc", "lp", "plc", "the", "and", "et", "of", "smt", "shri", "sri", "m", "s", "ms", "dba", ""}
STREET = {"street": "st", "road": "rd", "lane": "ln", "avenue": "ave", "av": "ave", "drive": "dr",
          "court": "ct", "circle": "cir", "boulevard": "blvd", "bd": "blvd", "bld": "blvd", "place": "pl",
          "highway": "hwy", "parkway": "pkwy", "terrace": "ter", "trail": "trl", "square": "sq",
          "rue": "r", "allee": "all", "chemin": "ch", "impasse": "imp", "route": "rte", "north": "n",
          "south": "s", "east": "e", "west": "w", "calcutta": "kolkata", "bombay": "mumbai",
          "bangalore": "bengaluru", "madras": "chennai", "gurgaon": "gurugram", "null": "", "na": "",
          "none": "", "no": "", "bis": "", "nd": "", "th": "", "rd_": "", "st_": "", "first": "1", "second": "2",
          "third": "3", "fourth": "4", "fifth": "5", "sixth": "6", "seventh": "7", "eighth": "8", "ninth": "9",
          "tenth": "10", "saint": "st", "mount": "mt", "fort": "ft", "nagara": "nagar"}
US_STATES = {"alabama": "al", "alaska": "ak", "arizona": "az", "arkansas": "ar", "california": "ca",
             "colorado": "co", "connecticut": "ct", "delaware": "de", "florida": "fl", "georgia": "ga",
             "hawaii": "hi", "idaho": "id", "illinois": "il", "indiana": "in", "iowa": "ia", "kansas": "ks",
             "kentucky": "ky", "louisiana": "la", "maine": "me", "maryland": "md", "massachusetts": "ma",
             "michigan": "mi", "minnesota": "mn", "mississippi": "ms", "missouri": "mo", "montana": "mt",
             "nebraska": "ne", "nevada": "nv", "new hampshire": "nh", "new jersey": "nj", "new mexico": "nm",
             "new york": "ny", "north carolina": "nc", "north dakota": "nd", "ohio": "oh", "oklahoma": "ok",
             "oregon": "or", "pennsylvania": "pa", "rhode island": "ri", "south carolina": "sc",
             "south dakota": "sd", "tennessee": "tn", "texas": "tx", "utah": "ut", "vermont": "vt",
             "virginia": "va", "washington": "wa", "west virginia": "wv", "wisconsin": "wi", "wyoming": "wy",
             "district of columbia": "dc"}
IN_STATES = {"andhra pradesh": "ap", "arunachal pradesh": "ar", "assam": "as", "bihar": "br",
             "chhattisgarh": "cg", "goa": "ga", "gujarat": "gj", "haryana": "hr", "himachal pradesh": "hp",
             "jharkhand": "jh", "karnataka": "ka", "kerala": "kl", "madhya pradesh": "mp", "maharashtra": "mh",
             "manipur": "mn", "meghalaya": "ml", "mizoram": "mz", "nagaland": "nl", "odisha": "od",
             "orissa": "od", "punjab": "pb", "rajasthan": "rj", "sikkim": "sk", "tamil nadu": "tn",
             "telangana": "tg", "tripura": "tr", "uttar pradesh": "up", "uttarakhand": "uk",
             "west bengal": "wb", "delhi": "dl", "new delhi": "dl", "jammu and kashmir": "jk",
             "chandigarh": "ch", "puducherry": "py", "pondicherry": "py"}
PHRASES = {f" {k} ": f" {v} " for k, v in {**US_STATES, **IN_STATES}.items() if " " in k}
ASTOP = ["floor", "ground", "first", "second", "unit", "apartment", "suite", "city", "near", "plot",
         "sector", "shop", "site", "behind", "opposite", "building", "main", "cross", "nagar", "village",
         "town", "estate", "colony", "house", "office", "door", "flat", "block", "road", "street"]
WORDS = {**STREET, **{k: v for k, v in {**US_STATES, **IN_STATES}.items() if " " not in k}}


ANUSVARA = re.compile(r"m(?=[^aeiouybpm\W])")  # anyascii writes anusvara as 'm': kmsltemts -> knsltents


def _tr(x):
    return ANUSVARA.sub("n", anyascii(x).lower()) if x and not x.isascii() else x


def translit(c):
    """ASCII-fold everything (Indic scripts, accents). ASCII strings pass through untouched."""
    return c.map_batches(lambda s: pl.Series([_tr(x) for x in s.to_list()], dtype=pl.Utf8), return_dtype=pl.Utf8)


def base(c):
    return translit(c).str.to_lowercase().str.replace_all("&", " and ", literal=True)


def tokens(c):
    return c.str.replace_all(r"[^a-z0-9]+", " ").str.strip_chars().str.split(" ")


def skel_expr(c):
    """Phonetic consonant skeleton per token (handles translit vowel loss: laxmi/lksmi -> lksm)."""
    return (c.str.replace_all("x", "ks").str.replace_all("ph", "f").str.replace_all("q", "k")
            .str.replace_all("c", "k").str.replace_all("w", "v").str.replace_all("z", "j")
            .str.replace_all("g", "j").str.replace_all(r"\B[aeiouyh]+", "").str.replace_many([c + c for c in "bdfjklmnprstv"], list("bdfjklmnprstv"))
            .str.replace_many([c + c for c in "bdfjklmnprstv"], list("bdfjklmnprstv")))


def normalize(df):
    name = (base(pl.col("name"))
            .str.replace_all(r"\(\s*id:?\s*\d+\s*\)|\[?#\s*\d+\]?|\+?\d[\d\- ]{8,}\d", " ")  # ids, #123, phones
            .str.replace_all(r"https?://|www\.|\.(com|in|net|org|co|fr|biz|io|us)\b|^@", " ")
            .str.replace_all(r"([a-z])\.([a-z])\.", "$1$2")   # l.l.p. -> llp
            .str.replace_all(r"([a-z])\.([a-z])\.", "$1$2")
            .str.replace_all(".", "", literal=True).str.replace_all("'", "", literal=True))
    ntok = tokens(name).list.eval(pl.element().replace(LEGAL_MAP))
    ntok = ntok.list.eval(pl.element().filter(pl.element() != ""))
    core_l = ntok.list.eval(pl.element().filter(~pl.element().is_in(list(LEGAL))))
    addr = (" " + base(pl.col("addr")).str.replace_all(r"[^a-z0-9]+", " ") + " ")
    addr = addr.str.replace_many(list(PHRASES), list(PHRASES.values()))
    atok = tokens(addr).list.eval(pl.element().replace(WORDS)).list.eval(
        pl.element().filter(pl.element() != ""))
    return df.select(
        "id", "src", "country",
        ntok.list.join(" ").alias("name_n"),
        core_l.list.unique().list.sort().list.join(" ").alias("core"),
        core_l.list.join("").alias("nosp"),
        core_l.list.eval(skel_expr(pl.element())).list.unique().list.sort().list.join(" ").alias("skel"),
        atok.list.join(" ").alias("addr_n"),
        addr.str.extract_all(r"\d+").list.eval(pl.element().str.strip_chars_start("0"))
            .list.eval(pl.element().filter(pl.element() != "")).list.unique(maintain_order=True).list.head(5).alias("nums"),
        atok.list.eval(pl.element().filter(pl.element().str.contains(r"^[a-z]{4,}$") & ~pl.element().is_in(ASTOP)))
            .list.unique().list.head(6).alias("atoks"),
        core_l.list.eval(skel_expr(pl.element())).list.eval(pl.element().filter(pl.element().str.len_chars() >= 3))
            .list.unique().alias("stoks"),
    )


def main():
    for split in SPLITS:
        out = WORK / f"{split}_norm2.parquet"
        if done(out):
            continue
        with timer(f"normalize {split}"):
            normalize(pl.scan_parquet(WORK / f"{split}_records.parquet")).sink_parquet(out)


if __name__ == "__main__":
    import sys
    if "--demo" in sys.argv:
        pl.Config.set_fmt_str_lengths(60); pl.Config.set_tbl_width_chars(300); pl.Config.set_tbl_rows(30)
        d = pl.read_parquet(WORK / "train_records.parquet", n_rows=2_000_000).sample(25, seed=int(sys.argv[-1]))
        print(pl.concat([d.select("name", "addr"), normalize(d.lazy()).collect()
                         .select("core", "skel", "nosp", "addr_n", "nums")], how="horizontal"))
    else:
        main()

In [ ]:
%%writefile /kaggle/working/src/s2_block.py
"""Stage 2: blocking. Keys (all within country): see keys(). Blocks bigger than caps are dropped (generic keys).
Out per S1 bucket: [s1, id, nk0..nk8 = #shared keys per type, sc, rk, ncand_raw], top-K per S1."""
import sys

import polars as pl

from common import SPLITS, WORK, done, timer

CAP1, CAP23 = 20, 100   # max S1 / S2+S3 records sharing a key
PARTS = 32              # key-hash partitions -> bounded RAM
NKT = 9
KW = [3, 1, 1, 1, 1, 3, 3, 1, 1]  # rank weight per key type


RARE = {"atoks": 3, "ntoks": 3, "stoks": 2}
KCHUNKS = 16  # key emission in id chunks -> bounded RAM


def ntoks(lf):
    return lf.with_columns(pl.col("core").str.split(" ").list.eval(
        pl.element().filter(pl.element().str.len_chars() >= 3)).alias("ntoks"))


def token_df(lf, col):
    """Document frequency per (country, token) over the whole split; tokens seen once are useless keys."""
    return (lf.select("id", "country", pl.col(col).list.unique().alias("t")).explode("t").drop_nulls("t")
            .group_by("country", "t").len("df").filter(pl.col("df") >= 2).collect())


def rare(lf, col, n, df):
    """Per record keep the n rarest tokens of list column `col`."""
    t = lf.select("id", "country", pl.col(col).list.unique().alias("t")).explode("t").drop_nulls("t")
    return (t.join(df.lazy(), on=["country", "t"]).sort("id", "df", "t").group_by("id")
            .agg(pl.col("t").head(n).alias(col)))


def keys(lf, dfs):
    """kt: 0 core name | 1 number x rare addr token | 2 rare name tok x rare addr tok | 3 rare addr token pair
           4 rare name token pair | 5 name skeleton | 6 name without spaces (web/handle names)
           7 rare skeleton tok x rare addr tok | 8 rare skeleton tok x number"""
    lf = ntoks(lf)
    base = lf.select("id", "src", "country", "core", "skel", "nosp", pl.col("nums").list.head(3))
    for col, n in RARE.items():
        base = base.join(rare(lf, col, n, dfs[col]), on="id", how="left")

    def emit(kt, cols, frame):
        return frame.select("id", "src", pl.concat_str(["country", pl.lit(str(kt)), *cols], separator="|")
                            .hash().alias("k"), pl.lit(kt, pl.Int8).alias("kt"))

    def pairs(col):
        a = base.select("id", "src", "country", pl.col(col).alias("a")).explode("a")
        b = base.select("id", pl.col(col).alias("b")).explode("b")
        return a.join(b, on="id").filter(pl.col("a") < pl.col("b"))

    def cross(c1, c2):
        return base.explode(c1).explode(c2).drop_nulls([c1, c2])

    return pl.concat([
        emit(0, ["core"], base.filter(pl.col("core") != "")),
        emit(1, ["nums", "atoks"], cross("nums", "atoks")),
        emit(2, ["ntoks", "atoks"], cross("ntoks", "atoks")),
        emit(3, ["a", "b"], pairs("atoks")),
        emit(4, ["a", "b"], pairs("ntoks")),
        emit(5, ["skel"], base.filter(pl.col("skel").str.len_chars() >= 4)),
        emit(6, ["nosp"], base.filter(pl.col("nosp").str.len_chars() >= 6)),
        emit(7, ["stoks", "atoks"], cross("stoks", "atoks")),
        emit(8, ["stoks", "nums"], cross("stoks", "nums")),
    ]).unique(["id", "k"])


BUCKETS = 8   # S1-id buckets: every later stage works one bucket at a time
TOPK = int(__import__("os").environ.get("BER_TOPK", 60))   # candidates kept per S1


def cand_path(split, j):
    return WORK / f"{split}_cand2_{j}.parquet"


def block(split):
    if all(done(cand_path(split, j)) for j in range(BUCKETS)):
        return
    kfs = [WORK / f"{split}_keys2_{c}.parquet" for c in range(KCHUNKS)]
    if not all(done(f) for f in kfs):
        lf = ntoks(pl.scan_parquet(WORK / f"{split}_norm2.parquet").with_columns(pl.col("country").cast(pl.Utf8)))
        with timer(f"token df {split}"):
            dfs = {col: token_df(lf, col) for col in RARE}
        for c, f in enumerate(kfs):
            if not done(f):
                with timer(f"keys {split} chunk {c}"):
                    keys(lf.filter(pl.col("id") % KCHUNKS == c), dfs).collect().write_parquet(f)
        del dfs
    kf = kfs
    parts = []
    for p in range(PARTS):
        pf = WORK / f"{split}_cand2_p{p}.parquet"
        if not done(pf):
            with timer(f"join {split} part {p}"):
                k = pl.scan_parquet(kf).filter(pl.col("k") % PARTS == p).collect()
                sz = k.group_by("k").agg((pl.col("src") == 1).sum().alias("n1"),
                                         (pl.col("src") != 1).sum().alias("n23"))
                good = sz.filter((pl.col("n1") >= 1) & (pl.col("n1") <= CAP1) &
                                 (pl.col("n23") >= 1) & (pl.col("n23") <= CAP23)).select("k")
                k = k.join(good, on="k", how="semi")
                a = k.filter(pl.col("src") == 1).select(pl.col("id").alias("s1"), "k", "kt")
                b = k.filter(pl.col("src") != 1).select("id", "k")
                (a.join(b, on="k").group_by("s1", "id")
                 .agg(*[(pl.col("kt") == t).sum().cast(pl.Int8).alias(f"nk{t}") for t in range(NKT)])
                 .write_parquet(pf))
                del k, a, b
        parts.append(pf)
    # merge per S1 bucket (a pair can come from several key partitions) -> rank -> top-K
    for j in range(BUCKETS):
        if done(cand_path(split, j)):
            continue
        with timer(f"merge {split} bucket {j}"):
            c = (pl.scan_parquet(parts).filter(pl.col("s1") % BUCKETS == j).group_by("s1", "id")
                 .agg(pl.col(f"nk{t}").sum() for t in range(NKT)).collect())
            c = c.with_columns(pl.sum_horizontal([KW[t] * pl.col(f"nk{t}").cast(pl.Int16) for t in range(NKT)])
                               .cast(pl.Int16).alias("sc"))
            c = c.with_columns(pl.col("sc").rank("ordinal", descending=True).over("s1").cast(pl.Int16).alias("rk"),
                               pl.len().over("s1").cast(pl.Int32).alias("ncand_raw"))
            c.filter(pl.col("rk") <= TOPK).write_parquet(cand_path(split, j))
            if split == "train":   # uncapped copy of a single bucket, to measure what top-K loses
                if j == 0:
                    c.select("s1", "id", "sc", "rk").write_parquet(WORK / "train_cand2_full0.parquet")
            del c
    for pf in parts:
        pf.unlink()


def recall_report():
    c = pl.scan_parquet([cand_path("train", j) for j in range(BUCKETS)])
    gt = pl.scan_parquet(WORK / "train_gt.parquet")
    n_s1 = pl.scan_parquet(WORK / "train_norm2.parquet").filter(pl.col("src") == 1).select(pl.len()).collect().item()
    npairs = c.select(pl.len()).collect().item()
    hit = gt.join(c, on=["s1", "id"], how="inner").select(pl.len()).collect().item()
    ngt = gt.select(pl.len()).collect().item()
    print(f"pairs {npairs:,}  per S1 {npairs / n_s1:.1f}  recall {hit / ngt:.4f} ({hit:,}/{ngt:,})")
    f = pl.read_parquet(WORK / "train_cand2_full0.parquet")
    g0 = gt.filter(pl.col("s1") % BUCKETS == 0).collect()
    h = g0.join(f, on=["s1", "id"])
    print("bucket0 uncapped recall", round(h.height / g0.height, 4),
          {k: round((h["rk"] <= k).sum() / g0.height, 4) for k in (10, 20, 30, 40, 50, 60)})


if __name__ == "__main__":
    for s in (sys.argv[1:] or SPLITS):
        block(s)
    recall_report()

In [ ]:
%%writefile /kaggle/working/src/s3_features.py
"""Stage 3: pair features, one S1 bucket per process (bounded RAM on a 12 GB box).
prep (once per split): {split}_norm3.parquet = norm2 cols + hashed token lists + nonascii flag + name frequencies,
                       {split}_idf_{ch,ah}.parquet = per-country IDF of name / address tokens.
bucket j: loads only the records its candidate pairs touch, computes features in sub-chunks.
Out: {split}_feat2_{j}.parquet (+ label y for train). Competition features across buckets come later (s5)."""
import sys

import numpy as np
import polars as pl
from rapidfuzz import fuzz
from rapidfuzz.distance import Levenshtein
from rapidfuzz.process import cpdist

from common import FEAT, WORK, done, timer
from s2_block import BUCKETS, cand_path

SUB = 16 # sub-chunks per bucket
STR = [  # (feature, column, scorer)
    ("core_tset", "core", fuzz.token_set_ratio),
    ("core_tsort", "core", fuzz.token_sort_ratio),
    ("core_ratio", "core", fuzz.ratio),
    ("core_partial", "core", fuzz.partial_ratio),
    ("name_ratio", "name_n", fuzz.ratio),
    ("skel_tset", "skel", fuzz.token_set_ratio),
    ("skel_ratio", "skel", fuzz.ratio),
    ("nosp_ratio", "nosp", fuzz.ratio),
    ("nosp_partial", "nosp", fuzz.partial_ratio),
    ("addr_tset", "addr_n", fuzz.token_set_ratio),
    ("addr_tsort", "addr_n", fuzz.token_sort_ratio),
    ("addr_partial", "addr_n", fuzz.partial_ratio),
    ("addr_ratio", "addr_n", fuzz.ratio),
]
COLS_N = ["name_n", "core", "skel", "nosp", "addr_n", "nums", "ch", "ah", "nonascii", "core_n1", "core_n23"]


def feat_path(split, j):
    return WORK / f"{split}_{FEAT}_{j}.parquet"


def norm3_path(split):
    return WORK / f"{split}_norm3.parquet"


def idf_path(split, col):
    return WORK / f"{split}_idf_{col}.parquet"


def prep(split):
    """Hashed token lists, IDF tables (counted in 8 id-chunks), nonascii flag, name frequencies."""
    if done(norm3_path(split)) and done(idf_path(split, "ah")):
        return
    with timer(f"prep {split}"):
        lf = pl.scan_parquet(WORK / f"{split}_norm2.parquet").with_columns(
            pl.col("core").str.split(" ").list.eval(pl.element().filter(pl.element() != "").hash())
            .list.unique().alias("ch"),
            pl.col("addr_n").str.split(" ").list.eval(pl.element().filter(pl.element() != "").hash())
            .list.unique().alias("ah"))
        cnt = lf.group_by("country").len("N").collect()
        for col in ("ch", "ah"):
            parts = [lf.filter(pl.col("id") % 8 == c).select("country", pl.col(col).alias("t")).explode("t")
                     .drop_nulls("t").group_by("country", "t").len("df").collect() for c in range(8)]
            (pl.concat(parts).group_by("country", "t").agg(pl.col("df").sum()).join(cnt, on="country")
             .select("country", "t", (pl.col("N") / pl.col("df")).log().cast(pl.Float32).alias("w"))
             .write_parquet(idf_path(split, col)))
            del parts
        n = lf.select("id", "src", "country", "name_n", "core", "skel", "nosp", "addr_n", "nums", "ch", "ah").collect()
        raw = pl.scan_parquet(WORK / f"{split}_records.parquet").select(
            pl.col("id").alias("rid"), pl.col("name").str.contains(r"[^\x00-\x7f]").alias("nonascii")).collect()
        assert (raw["rid"] == n["id"]).all(), "records / norm2 row order differs"
        n = n.with_columns(raw["nonascii"])   # same row order: no join copy
        del raw
        n = n.with_columns((pl.col("src") == 1).sum().over(["country", "core"]).cast(pl.Int32).alias("core_n1"),
                           (pl.col("src") != 1).sum().over(["country", "core"]).cast(pl.Int32).alias("core_n23"))
        n.write_parquet(norm3_path(split))


def wjacc(x, col, idf, name):
    """IDF-weighted Jaccard of hashed token lists col (S1) vs col_2, plus max IDF of a shared token."""
    r = pl.int_range(pl.len(), dtype=pl.UInt32).alias("r")
    xx = x.select(r, "country", col, f"{col}_2")
    L = xx.select("r", "country", pl.col(col).alias("t")).explode("t").drop_nulls("t").join(idf, on=["country", "t"])
    R = xx.select("r", "country", pl.col(f"{col}_2").alias("t")).explode("t").drop_nulls("t").join(idf, on=["country", "t"])
    sl = L.group_by("r").agg(pl.col("w").sum().alias("sl"))
    sr = R.group_by("r").agg(pl.col("w").sum().alias("sr"))
    sh = L.join(R.select("r", "t"), on=["r", "t"]).group_by("r").agg(pl.col("w").sum().alias("sh"),
                                                                    pl.col("w").max().alias("mx"))
    o = (xx.select("r").join(sl, on="r", how="left").join(sr, on="r", how="left").join(sh, on="r", how="left")
         .sort("r").fill_null(0))
    u = (o["sl"] + o["sr"] - o["sh"]).to_numpy()
    return {f"{name}_wj": (o["sh"].to_numpy() / np.maximum(u, 1e-6)).astype(np.float32),
            f"{name}_wsh": o["sh"].to_numpy().astype(np.float32),
            f"{name}_wmax": o["mx"].to_numpy().astype(np.float32),
            f"{name}_wcov1": (o["sh"].to_numpy() / np.maximum(o["sl"].to_numpy(), 1e-6)).astype(np.float32),
            f"{name}_wcov2": (o["sh"].to_numpy() / np.maximum(o["sr"].to_numpy(), 1e-6)).astype(np.float32)}


def first_num_dist(a, b):
    """Levenshtein distance between first house numbers (typo'd digits), 9 if either missing."""
    return np.fromiter((Levenshtein.distance(p, q) if p is not None and q is not None else 9 for p, q in zip(a, b)),
                       dtype=np.int8, count=len(a))


def features_chunk(c, n, idf):
    x = (c.join(n.select(pl.col("id").alias("s1"), "country", *COLS_N), on="s1")
         .join(n.select("id", "src", *COLS_N), on="id", suffix="_2"))
    feats = {}
    for f, col, scorer in STR:
        feats[f] = cpdist(x[col].to_list(), x[col + "_2"].to_list(), scorer=scorer, workers=-1, dtype=np.uint8)
    feats.update(wjacc(x, "ch", idf["ch"], "nm"))
    feats.update(wjacc(x, "ah", idf["ah"], "ad"))
    feats["num1_lev"] = first_num_dist(x["nums"].list.first().to_list(), x["nums_2"].list.first().to_list())
    num_i = x["nums"].list.set_intersection(x["nums_2"]).list.len()
    num_u = x["nums"].list.set_union(x["nums_2"]).list.len()
    return x.select(
        "s1", "id", pl.col("src").cast(pl.Int8), *[f"nk{t}" for t in range(9)], "sc", "rk", "ncand_raw",
        pl.col("core").str.len_chars().cast(pl.Int16).alias("len_core"),
        pl.col("core_2").str.len_chars().cast(pl.Int16).alias("len_core2"),
        pl.col("addr_n").str.len_chars().cast(pl.Int16).alias("len_addr1"),
        pl.col("addr_n_2").str.len_chars().cast(pl.Int16).alias("len_addr2"),
        pl.col("nums").list.len().cast(pl.Int8).alias("n_nums1"),
        pl.col("nums_2").list.len().cast(pl.Int8).alias("n_nums2"),
        pl.col("nonascii_2").alias("nonascii2"),
        (pl.col("core") == pl.col("core_2")).alias("core_eq"),
        (pl.col("skel") == pl.col("skel_2")).alias("skel_eq"),
        "core_n1", "core_n23", pl.col("core_n1_2").alias("core2_n1"),
        (pl.col("nums").list.first() == pl.col("nums_2").list.first()).fill_null(False).alias("num_first_eq"),
        pl.col("nums_2").list.contains(pl.col("nums").list.first()).fill_null(False).alias("num1_in_2"),
    ).with_columns(
        num_i.cast(pl.Int8).alias("num_inter"),
        (num_i / num_u.clip(lower_bound=1)).cast(pl.Float32).alias("num_jacc"),
        (num_u - num_i).cast(pl.Int8).alias("num_diff"),
        *[pl.Series(k, v) for k, v in feats.items()],
    )


def rss():
    try:
        import resource
        return f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB peak"
    except ImportError:   # Windows (local smoke tests)
        return "rss n/a"


def load_records(split, ids):
    """Rows of norm3 whose id is in ids. pyarrow's dataset scanner filters batch by batch, so only the
    result (~1.5 GB) is materialised (a full read + filter peaked near 10 GB, polars batches near 5.4 GB)."""
    import pyarrow as pa
    import pyarrow.dataset as ds
    t = ds.dataset(str(norm3_path(split)), format="parquet").to_table(
        filter=ds.field("id").isin(pa.array(ids["id"].to_numpy())), batch_size=200_000)
    return pl.from_arrow(t).with_columns(pl.col("country").cast(pl.Utf8))


def finish_chunk(out, gt):
    """Per-S1 relative features (chunks partition S1, so windows are exact) + label."""
    out = out.with_columns((pl.col("core_tset").cast(pl.Int16) + pl.col("addr_tset")).alias("pair_sum"),
                           (pl.col("skel_tset").cast(pl.Int16) + pl.col("addr_tset")).alias("pair_sum2"))
    out = out.with_columns(
        (pl.col("pair_sum").max().over("s1") - pl.col("pair_sum")).alias("gap_s1_best"),
        pl.col("pair_sum").rank("min", descending=True).over("s1").cast(pl.Int16).alias("rk_s1_ps"),
        (pl.col("pair_sum2").max().over("s1") - pl.col("pair_sum2")).alias("gap_s1_best2"),
    )
    if gt is not None:
        out = out.join(gt, on=["s1", "id"], how="left").with_columns(pl.col("y").fill_null(0))
    return out


def features(split, j):
    """Writes feat_path(split, j). Each sub-chunk is finished and spilled to disk, then stream-concatenated."""
    c = pl.read_parquet(cand_path(split, j))
    n = load_records(split, pl.concat([c["s1"], c["id"]]).unique().to_frame("id"))
    idf = {col: pl.read_parquet(idf_path(split, col)).with_columns(pl.col("country").cast(pl.Utf8))
           for col in ("ch", "ah")}
    print(f"  loaded {n.height:,} records for {c.height:,} pairs, {rss()}", flush=True)
    gt = None
    if split == "train":
        gt = (pl.scan_parquet(WORK / "train_gt.parquet").filter(pl.col("s1") % BUCKETS == j)
              .with_columns(pl.lit(1, pl.Int8).alias("y")).collect())
    parts = []
    for s in range(SUB):
        cs = c.filter((pl.col("s1") // BUCKETS) % SUB == s)
        pf = WORK / f"_tmp_{split}_{j}_{s}.parquet"
        finish_chunk(features_chunk(cs, n, idf), gt).write_parquet(pf)
        parts.append(pf)
        if s % 4 == 3:
            print(f"  chunk {s + 1}/{SUB} {rss()}", flush=True)
        del cs
    del n, c
    pl.scan_parquet(parts).sink_parquet(feat_path(split, j))
    for pf in parts:
        pf.unlink()


if __name__ == "__main__":
    # usage: s3_features.py <split> prep | s3_features.py <split> <bucket>
    split, what = sys.argv[1], sys.argv[2]
    if what == "prep":
        prep(split)
    else:
        j = int(what)
        if not done(feat_path(split, j)):
            with timer(f"features {split} {j}"):
                features(split, j)

In [ ]:
%%writefile /kaggle/working/src/s4_train.py
"""Stage 4: XGBoost (GPU) pair classifier with 2-fold out-of-fold predictions by S1 bucket.
stage 1: pair features only. stage 2: + competition features built from stage-1 OOF probabilities (s5_comp).
stage 3: stage 2 + cross-encoder logit on uncertain pairs (s8_ce; NaN = not scored) + optional embedding cosine (e2).
stage 5: same as stage 4 once the s14 address add-on (addr4_*) exists -> stage 4 + house-number-noise features.
stage 4: stage 1 features + cross-encoder logit, NO competition features (stacking on p1 proved fragile on test: v2
         stage 2 OOF +0.003 but LB -0.004).
Test-time consistency: a stacked model trained on fold f's buckets saw inputs produced by the *other* fold's model, so on
test each booster g gets the comp / ce inputs of model 1-g (variant files/columns "_m0"/"_m1"), not the 2-model mean.
Data is streamed half-bucket by half-bucket into a GPU QuantileDMatrix, so host RAM holds one piece at a time.
Out: WORK/p{stage}_{split}_{j}.parquet [s1, id, p]   (train = OOF, test = mean of the two fold models)."""
import json
import sys

import numpy as np
import polars as pl
import xgboost as xgb

from common import TAG, WORK, done, f05, timer
from s2_block import BUCKETS
from s3_features import feat_path

FOLDS = [[0, 2, 4, 6], [1, 3, 5, 7]]
DROP = {"s1", "id", "y"}
_E = __import__("os").environ.get
PARAMS = dict(objective="binary:logistic", eval_metric="logloss", tree_method="hist", device="cuda",
              max_bin=256, eta=float(_E("BER_ETA", 0.08)), grow_policy="lossguide", max_leaves=int(_E("BER_LEAVES", 255)),
              max_depth=0, min_child_weight=int(_E("BER_MCW", 20)), subsample=0.8,
              colsample_bytree=float(_E("BER_COLS", 0.8)), reg_lambda=1.0, seed=int(_E("BER_SEED", 0)))
ROUNDS, EARLY = 3000, 100
NPIECE = int(_E("BER_NPIECE", 4))   # each bucket is streamed in pieces: host RAM holds one piece at a time
EASY_P, EASY_KEEP = 0.01, int(_E("BER_EASY_KEEP", 10))   # training keeps hard pairs + 1/EASY_KEEP of easy ones (weight EASY_KEEP)
TRAIN_BUCKETS = int(_E("BER_TRAIN_BUCKETS", 4))   # buckets per fold used for training (2 = half data, for 12 GB machines)


def comp_path(split, j, v=None):
    """v = which stage-1 model produced the test p1 the comp features were built from (test only)."""
    return WORK / (f"comp{TAG}_{split}_{j}.parquet" if v is None else f"comp{TAG}_{split}_m{v}_{j}.parquet")


def uses_comp(stage):
    return stage in (2, 3)


def ce_path(split, j):
    return WORK / f"ce{TAG}_{split}_{j}.parquet"


def embcos_path(split, j):
    return WORK / f"embcos{TAG}_{split}_{j}.parquet"


def idc_path(split, j):
    """s16 per-record competition features (similarity-based, untagged)."""
    return WORK / f"idc_{split}_{j}.parquet"


def addr_path(split, j):
    """s14 address add-on (model-independent, untagged)."""
    return WORK / f"addr4_{split}_{j}.parquet"


def ftc_path(split, j):
    """s18 fold-wise fine-tuned e5 cosine features (run E; test has per-model _m0/_m1 columns, untagged)."""
    return WORK / f"ftc_{split}_{j}.parquet"


def extra_paths(split, j, stage):
    """Optional stage-3 feature files (joined when present for train bucket 0, i.e. produced for all buckets)."""
    if stage < 3:   # stage 3 = comp + extras, stage 4 = extras only
        return []
    return [f(split, j) for f in (ce_path, embcos_path, addr_path, idc_path, ftc_path) if f("train", 0).exists()]


def p_path(stage, split, j):
    return WORK / f"p{stage}{TAG}_{split}_{j}.parquet"


def model_path(stage, f):
    return WORK / f"xgb_s{stage}{TAG}_f{f}.json"


def frame(split, j, stage, piece=None, sample=False, variant=None):
    """Features of bucket j (optionally one quarter), + comp (stages 2-3) / extra features (stages 3-4).
    variant (test only): use the comp file / ce_m{variant} column made from model `variant`.
    sample=True (training only): drop most easy negatives (p1 < EASY_P); returns (df, weights)."""
    def part(path):
        lf = pl.scan_parquet(path)
        return lf if piece is None else lf.filter((pl.col("s1") // BUCKETS) % NPIECE == piece)
    lf = part(feat_path(split, j))
    if uses_comp(stage):
        lf = lf.join(part(comp_path(split, j, variant)), on=["s1", "id"], how="left")
    for ep in extra_paths(split, j, stage):
        e = part(ep)
        names = e.collect_schema().names()
        vcols = [c for c in names if c[-3:] in ("_m0", "_m1")]
        if variant is not None:   # swap in the per-model column, e.g. ce <- ce_m1
            e = e.with_columns([pl.col(f"{c}_m{variant}").alias(c) for c in names if f"{c}_m{variant}" in names])
        lf = lf.join(e.drop(vcols), on=["s1", "id"], how="left")
    if not sample:
        return lf.collect()
    if not uses_comp(stage):
        # no p1 yet: easy negatives = outside the S1's top-10 by fuzzy pair score AND by embedding cosine (true matches
        # are nearly always inside); keep all others + 1/EASY_KEEP of the easy ones (weight EASY_KEEP). v3 has 139M pairs.
        names = lf.collect_schema().names()
        easy = pl.col("rk_s1_ps") > 10
        if "emb_rk" in names:
            easy = easy & (pl.col("emb_rk") > 10)
        df = lf.with_columns(easy.alias("_easy")).filter(
            ~pl.col("_easy") | (pl.struct("s1", "id").hash(seed=7) % EASY_KEEP == 0)).collect()
        w = np.where(df["_easy"].to_numpy(), float(EASY_KEEP), 1.0).astype(np.float32)
        return df.drop("_easy"), w
    easy = pl.col("p1") < EASY_P
    df = lf.filter(~easy | (pl.struct("s1", "id").hash(seed=7) % EASY_KEEP == 0)).collect()
    return df, np.where(df["p1"].to_numpy() < EASY_P, float(EASY_KEEP), 1.0).astype(np.float32)


def feature_cols(stage):
    cols = pl.scan_parquet(feat_path("train", 0)).collect_schema().names()
    if uses_comp(stage):
        cols += [c for c in pl.scan_parquet(comp_path("train", 0)).collect_schema().names() if c not in cols]
    for ep in extra_paths("train", 0, stage):
        cols += [c for c in pl.scan_parquet(ep).collect_schema().names() if c not in cols and c[-3:] not in ("_m0", "_m1")]
    return [c for c in cols if c not in DROP]


class Batches(xgb.DataIter):
    def __init__(self, buckets, stage, cols):
        self.pieces = [(j, h) for j in buckets for h in range(NPIECE)]
        self.stage, self.cols, self.i = stage, cols, 0
        super().__init__()

    def next(self, input_data):
        if self.i == len(self.pieces):
            return False
        j, h = self.pieces[self.i]
        df, w = frame("train", j, self.stage, h, sample=True)
        X = df.select(pl.col(self.cols).cast(pl.Float32)).to_numpy()   # direct float32: no float64 temp copy
        y = df["y"].to_numpy()
        del df
        input_data(data=X, label=y, weight=w, feature_names=self.cols)
        del X, y, w
        self.i += 1
        return True

    def reset(self):
        self.i = 0


def predict(bsts, split, j, stage, cols):
    """Probability, a quarter bucket at a time. One booster -> p. Two boosters (test) -> p = mean, plus p_m0 / p_m1
    per model; for stacked stages booster g reads the inputs made by model 1-g (see module doc)."""
    stacked = split == "test" and len(bsts) == 2 and (uses_comp(stage) or bool(extra_paths(split, j, stage)))
    out = []
    for h in range(NPIECE):
        if stacked:
            ps = []
            for g, b in enumerate(bsts):
                df = frame(split, j, stage, h, variant=1 - g)
                X = xgb.DMatrix(df.select(pl.col(cols).cast(pl.Float32)).to_numpy(), feature_names=cols)
                ps.append(b.predict(X))
                ids = df.select("s1", "id")
                del df, X
        else:
            df = frame(split, j, stage, h)
            X = xgb.DMatrix(df.select(pl.col(cols).cast(pl.Float32)).to_numpy(), feature_names=cols)
            ps = [b.predict(X) for b in bsts]
            ids = df.select("s1", "id")
            del df, X
        o = ids.with_columns(pl.Series("p", np.mean(ps, axis=0)).cast(pl.Float32))
        if len(ps) == 2:
            o = o.with_columns(pl.Series("p_m0", ps[0]).cast(pl.Float32), pl.Series("p_m1", ps[1]).cast(pl.Float32))
        out.append(o)
    return pl.concat(out)


def train(stage):
    cols = feature_cols(stage)
    print(f"stage {stage}: {len(cols)} features", flush=True)
    for f, tr_b in enumerate(FOLDS):
        mp = model_path(stage, f)
        if done(mp):
            continue
        with timer(f"train stage {stage} fold {f} on buckets {tr_b}"):
            dtr = xgb.QuantileDMatrix(Batches(tr_b[:TRAIN_BUCKETS], stage, cols), max_bin=PARAMS["max_bin"])
            ev, ew = frame("train", FOLDS[1 - f][0], stage, 0, sample=True)   # quarter bucket, other fold
            dev = xgb.QuantileDMatrix(ev.select(pl.col(cols).cast(pl.Float32)).to_numpy(), label=ev["y"].to_numpy(),
                                      weight=ew, ref=dtr, feature_names=cols)
            del ev
            bst = xgb.train(PARAMS, dtr, ROUNDS, evals=[(dev, "ev")], early_stopping_rounds=EARLY,
                            verbose_eval=100)
            print("best iteration", bst.best_iteration, flush=True)
            bst.save_model(str(mp))
            imp = sorted(bst.get_score(importance_type="gain").items(), key=lambda kv: -kv[1])[:20]
            print("top gain:", [(k, int(v)) for k, v in imp], flush=True)
            del dtr, dev, bst
    return cols


def predict_all(stage, cols):
    bsts = [xgb.Booster(model_file=str(model_path(stage, f))) for f in range(2)]
    for b in bsts:
        b.set_param({"device": "cuda"})
    for f, va_b in enumerate(FOLDS):          # OOF: model trained on the other fold
        for j in va_b:
            if not done(p_path(stage, "train", j)):
                with timer(f"oof stage {stage} bucket {j}"):
                    predict([bsts[1 - f]], "train", j, stage, cols).write_parquet(p_path(stage, "train", j))
    for j in range(BUCKETS):
        if not done(p_path(stage, "test", j)):
            with timer(f"test stage {stage} bucket {j}"):
                predict(bsts, "test", j, stage, cols).write_parquet(p_path(stage, "test", j))


def assign(pred, t):
    """Each S2/S3 id -> its single best S1 (GT: every id matches at most one S1); keep if p >= t."""
    return (pred.filter(pl.col("p") >= t).sort("p", descending=True)
            .unique("id", keep="first").select("s1", "id"))


def evaluate(stage):
    pred = pl.concat([pl.read_parquet(p_path(stage, "train", j)).filter(pl.col("p") >= 0.15) for j in range(BUCKETS)])
    truth = pl.read_parquet(WORK / "train_gt.parquet")
    s1 = (pl.scan_parquet(WORK / "train_norm2.parquet").filter(pl.col("src") == 1)
          .select(pl.col("id").alias("s1"), "country").collect())
    res = {}
    for t in np.arange(0.2, 0.96, 0.025):
        res[round(float(t), 3)] = f05(s1, assign(pred, t), truth)
    best = max(res, key=res.get)
    print(f"stage {stage} OOF macro F0.5 (all train S1, global assign): best t={best} F={res[best]:.5f}")
    print({k: round(v, 4) for k, v in res.items()})
    by = {}
    a = assign(pred, best)
    for c in s1["country"].unique().to_list():
        sc = s1.filter(pl.col("country") == c)
        by[c] = f05(sc, a.join(sc, on="s1", how="semi"), truth.join(sc, on="s1", how="semi"))
    print("by country", by)
    json.dump({"t": best, "f05": res[best], "curve": res, "by_country": by},
              open(WORK / f"val_stage{stage}{TAG}.json", "w"))
    return best


if __name__ == "__main__":
    st = int(sys.argv[1]) if len(sys.argv) > 1 else 1
    c = train(st)
    predict_all(st, c)
    evaluate(st)

In [ ]:
%%writefile /kaggle/working/src/s5_comp.py
"""Stage 5: competition features from stage-1 probabilities (train: OOF, test: fold-mean).
Each S2/S3 id belongs to at most one S1 and each S1 has a handful of true ids, so a pair's p relative to its
rivals (same id across ALL S1 buckets; same S1) is strong evidence. Out: comp_{split}_{j}.parquet."""
import sys

import polars as pl

from common import WORK, done, timer
from s2_block import BUCKETS
from s4_train import comp_path, p_path


def build(split, stage=1, variant=None):
    """variant (test only): build from p_m{variant} = the test predictions of stage-1 model `variant`."""
    if all(done(comp_path(split, j, variant)) for j in range(BUCKETS)):
        return
    pcol = "p" if variant is None else f"p_m{variant}"
    with timer(f"comp {split}" + ("" if variant is None else f" from model {variant}")):
        src = [pl.scan_parquet(p_path(stage, split, j)).select("s1", "id", pl.col(pcol).alias("p")) for j in range(BUCKETS)]
        # per id across every S1 bucket: best / second-best p, number of candidate S1s
        ids = (pl.concat(src).group_by("id")
               .agg(pl.col("p").top_k(2).alias("top"), pl.len().cast(pl.Int16).alias("id_n"),
                    (pl.col("p") > 0.1).sum().cast(pl.Int16).alias("id_n10"))
               .select("id", "id_n", "id_n10", pl.col("top").list.get(0).alias("id_max"),
                       pl.col("top").list.get(1, null_on_oob=True).fill_null(0.0).alias("id_2nd"))
               .collect())
        for j in range(BUCKETS):
            p = (pl.read_parquet(p_path(stage, split, j)).select("s1", "id", pl.col(pcol).alias("p"))
                 .join(ids, on="id", how="left"))
            top = pl.col("p") >= pl.col("id_max")
            (p.with_columns(
                top.alias("id_top"),
                pl.when(top).then(pl.col("id_2nd")).otherwise(pl.col("id_max")).alias("id_rival"),
                pl.col("p").rank("ordinal", descending=True).over("s1").cast(pl.Int16).alias("s1_rank"),
                pl.col("p").max().over("s1").alias("s1_max"),
                pl.col("p").sum().over("s1").cast(pl.Float32).alias("s1_sum"),
                (pl.col("p") > 0.5).sum().over("s1").cast(pl.Int16).alias("s1_n50"))
             .with_columns((pl.col("p") - pl.col("id_rival")).alias("id_margin"),
                           (pl.col("s1_max") - pl.col("p")).alias("s1_gap"))
             .rename({"p": "p1"}).drop("id_max", "id_2nd")
             .write_parquet(comp_path(split, j, variant)))
            del p
        del ids


if __name__ == "__main__":
    for s in (sys.argv[1:] or ("train", "test")):
        if s == "test":
            build(s, variant=0)
            build(s, variant=1)
        else:
            build(s)

In [ ]:
%%writefile /kaggle/working/src/s6_write.py
"""Stage 6: final decode on test + write both submission TSVs.
matching_results: global assignment (each id -> best S1) with p >= threshold tuned on train OOF.
candidate_pairs: every pair the model scored (the exact inference set)."""
import json
import sys

import polars as pl

from common import OUT, TAG, WORK, decode_ids, timer
from s2_block import BUCKETS
from s4_train import assign, p_path


def to_tsv(s1_all, pairs, col, path):
    g = pairs.sort("s1", "id").group_by("s1", maintain_order=True).agg(decode_ids("id").str.join(",").alias(col))
    (s1_all.join(g, on="s1", how="left").with_columns(pl.col(col).fill_null(""))
     .select(decode_ids("s1").alias("source1_entity_id"), col)
     .write_csv(path, separator="\t", quote_style="never"))


def main(stage):
    t = json.load(open(WORK / f"val_stage{stage}{TAG}.json"))["t"]
    OUT.mkdir(parents=True, exist_ok=True)
    s1_all = (pl.scan_parquet(WORK / "test_norm2.parquet").filter(pl.col("src") == 1)
              .select(pl.col("id").alias("s1")).sort("s1").collect())
    with timer(f"write matches stage {stage} t={t}"):
        p = pl.concat([pl.read_parquet(p_path(stage, "test", j)).filter(pl.col("p") >= t) for j in range(BUCKETS)])
        match = assign(p, t)
        to_tsv(s1_all, match, "matched_entity_ids", OUT / "matching_results.tsv")
        print("matched pairs", match.height, "S1 with >=1 match", match["s1"].n_unique(), "of", s1_all.height)
        del p, match
    with timer("write candidates"):
        cand = pl.concat([pl.read_parquet(p_path(stage, "test", j), columns=["s1", "id"]) for j in range(BUCKETS)])
        to_tsv(s1_all, cand, "candidate_entity_ids", OUT / "candidate_pairs.tsv")


if __name__ == "__main__":
    main(int(sys.argv[1]) if len(sys.argv) > 1 else 2)

In [ ]:
%%writefile /kaggle/working/src/s8_ce.py
"""Stage 8: cross-encoder re-ranker on uncertain pairs (stage-1 OOF p in [LO, HI]).
Fine-tunes cross-encoder/mmarco-mMiniLMv2-L12-H384-v1 (Apache-2.0) once per fold, on the same S1-bucket folds as
XGBoost, so train scores are out-of-fold. Test scores = mean of the two fold models.
Input text per record: raw "name | address" (native scripts kept; the model is multilingual).
Out: ce_{split}_{j}.parquet [s1, id, ce] (logit, Float32). Pairs outside [LO, HI] are not scored (NaN in stage 3).
Resumable: fold models and per-bucket outputs are skipped when present."""
import math
import os
import sys
import time

import numpy as np
import polars as pl
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

from common import WORK, done, timer
from s2_block import BUCKETS
from s4_train import FOLDS, ce_path, p_path

MODEL = os.environ.get("BER_CE_MODEL", "/content/drive/MyDrive/MLC'26/models/mmarco-mMiniLMv2-L12-H384-v1")
LO, HI = float(os.environ.get("BER_CE_LO", 0.03)), float(os.environ.get("BER_CE_HI", 0.97))
MAX_TRAIN = int(os.environ.get("BER_CE_TRAIN", 1_500_000))   # training pairs per fold
MAXLEN = int(os.environ.get("BER_CE_MAXLEN", 96))
BS, BS_INF, LR = 64, 512, 3e-5
DEV = "cuda" if torch.cuda.is_available() else "cpu"


def fold_dir(f):
    return WORK / f"ce_model_f{f}"


def texts(split):
    return (pl.scan_parquet(WORK / f"{split}_records.parquet")
            .select("id", (pl.col("name") + " | " + pl.col("addr")).str.slice(0, 300).alias("t")))


def uncertain(split, j):
    return pl.scan_parquet(p_path(1, split, j)).filter(pl.col("p").is_between(LO, HI)).select("s1", "id")


def with_text(pairs, split):
    t = texts(split)
    return (pairs.join(t.rename({"id": "s1", "t": "a"}), on="s1").join(t.rename({"t": "b"}), on="id")
            .collect())


def train_fold(f):
    if done(fold_dir(f) / "config.json"):
        return
    with timer(f"ce train fold {f}"):
        gt = pl.scan_parquet(WORK / "train_gt.parquet").with_columns(pl.lit(1.0, pl.Float32).alias("y"))
        pairs = (pl.concat([uncertain("train", j) for j in FOLDS[f]]).join(gt, on=["s1", "id"], how="left")
                 .with_columns(pl.col("y").fill_null(0.0)))
        n = pairs.select(pl.len()).collect().item()
        pairs = with_text(pairs, "train").sample(min(n, MAX_TRAIN), seed=f, shuffle=True)
        print(f"fold {f}: {n:,} uncertain pairs, training on {pairs.height:,}, pos rate {pairs['y'].mean():.3f}",
              flush=True)
        tok = AutoTokenizer.from_pretrained(MODEL)
        model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=1).to(DEV)
        steps = math.ceil(pairs.height / BS)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
        sch = get_linear_schedule_with_warmup(opt, int(0.05 * steps), steps)
        scaler = torch.amp.GradScaler(enabled=DEV == "cuda")
        lossf = torch.nn.BCEWithLogitsLoss()
        a, b, y = pairs["a"].to_list(), pairs["b"].to_list(), pairs["y"].to_numpy()
        del pairs
        model.train()
        t0, run = time.time(), 0.0
        for k in range(steps):
            sl = slice(k * BS, (k + 1) * BS)
            enc = tok(a[sl], b[sl], padding=True, truncation=True, max_length=MAXLEN, return_tensors="pt").to(DEV)
            with torch.autocast(DEV, dtype=torch.float16, enabled=DEV == "cuda"):
                logit = model(**enc).logits.squeeze(-1)
            loss = lossf(logit.float(), torch.from_numpy(y[sl]).to(DEV))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sch.step()
            run = 0.98 * run + 0.02 * loss.item() if k else loss.item()
            if k % 2000 == 0:
                el = time.time() - t0
                print(f"  step {k}/{steps} loss {run:.4f} {(k + 1) * BS / max(el, 1e-6):.0f} pairs/s "
                      f"eta {el / (k + 1) * (steps - k - 1) / 60:.0f} min", flush=True)
        model.save_pretrained(fold_dir(f))
        tok.save_pretrained(fold_dir(f))
        del model, opt


@torch.no_grad()
def score(models, tok, df):
    """Logit of each model, [n_models, n]; pairs sorted by length so batches carry little padding."""
    order = np.argsort((df["a"].str.len_chars() + df["b"].str.len_chars()).to_numpy())
    a, b = df["a"].to_list(), df["b"].to_list()
    out = np.empty((len(models), df.height), dtype=np.float32)
    for k in range(0, df.height, BS_INF):
        idx = order[k:k + BS_INF]
        enc = tok([a[i] for i in idx], [b[i] for i in idx], padding=True, truncation=True, max_length=MAXLEN,
                  return_tensors="pt").to(DEV)
        with torch.autocast(DEV, dtype=torch.float16, enabled=DEV == "cuda"):
            for g, m in enumerate(models):
                out[g, idx] = m(**enc).logits.squeeze(-1).float().cpu().numpy()
    return out


def predict_all():
    tok = AutoTokenizer.from_pretrained(fold_dir(0))
    models = [AutoModelForSequenceClassification.from_pretrained(fold_dir(f)).to(DEV).half().eval() for f in (0, 1)]
    jobs = [("train", j, [models[1 - f]]) for f in (0, 1) for j in FOLDS[f]] + [("test", j, models) for j in range(BUCKETS)]
    for split, j, ms in jobs:
        if done(ce_path(split, j)):
            continue
        with timer(f"ce predict {split} {j}"):
            df = with_text(uncertain(split, j), split)
            t0 = time.time()
            ce = score(ms, tok, df)
            print(f"  {df.height:,} pairs, {df.height / max(time.time() - t0, 1e-6):.0f} pairs/s", flush=True)
            tmp = ce_path(split, j).with_suffix(".tmp")   # write + rename: a synced copy is never half-written
            o = df.select("s1", "id").with_columns(pl.Series("ce", ce.mean(0)))
            if len(ms) == 2:   # test: per-model logits, so stacked booster g can read model 1-g (see s4_train doc)
                o = o.with_columns(pl.Series("ce_m0", ce[0]), pl.Series("ce_m1", ce[1]))
            o.write_parquet(tmp)
            os.replace(tmp, ce_path(split, j))
            del df


def count():
    for split in ("train", "test"):
        n = sum(uncertain(split, j).select(pl.len()).collect().item() for j in range(BUCKETS))
        print(f"{split}: {n:,} uncertain pairs in [{LO}, {HI}]", flush=True)


if __name__ == "__main__":
    what = sys.argv[1] if len(sys.argv) > 1 else "all"
    if what in ("count", "all"):
        count()
    if what in ("train", "all"):
        train_fold(0)
        train_fold(1)
    if what in ("predict", "all"):
        predict_all()

In [ ]:
%%writefile /kaggle/working/src/s9_decode.py
"""Stage 9: per-S1 expected-F0.5 decoding (replaces one global threshold).
Macro F0.5 is averaged per S1, so the best prediction differs per S1: after the global assignment (each id -> its
best S1), sort that S1's candidates by p and keep the top-k that maximises
    E[F0.5 | k] ~= 1.25 * sum_{i<=k} q_i / (0.25 * sum_i q_i + k)     (k >= 1),   q = calibrated p
and predict nothing when P(no true match) = prod_i (1 - q_i) beats every k (singletons score 1.0 only when empty).
Calibration q = p ** g plus a floor f (pairs below f are ignored) are tuned on train OOF, never on the LB.
usage: s9_decode.py <stage>   -> writes WORK/val_decode{stage}.json and OUT(_ef) submission files."""
import json
import os
import sys

import numpy as np
import polars as pl

from common import OUT, TAG, WORK, f05, timer
from s2_block import BUCKETS
from s4_train import p_path
from s6_write import to_tsv

GAMMAS = [1.0, 1.25, 1.5]
FLOORS = [0.05, 0.15, 0.3]   # + the tuned global threshold t (added in main), so the search includes "t, then prune"


def assigned(pred, floor):
    """Global assignment: each id kept only for its best S1, and only if p >= floor."""
    return pred.filter(pl.col("p") >= floor).sort("p", descending=True).unique("id", keep="first")


def decode(pa, g):
    """pa: assigned pairs [s1, id, p]. Returns the kept pairs [s1, id]."""
    x = (pa.with_columns((pl.col("p").cast(pl.Float64) ** g).clip(1e-6, 1 - 1e-6).alias("q"))
         .sort(["s1", "q"], descending=[False, True])
         .with_columns(pl.col("q").cum_sum().over("s1").alias("cq"),
                       pl.int_range(1, pl.len() + 1).over("s1").alias("k"),
                       pl.col("q").sum().over("s1").alias("sq"),
                       (1 - pl.col("q")).log().sum().over("s1").exp().alias("p_empty")))
    x = x.with_columns((1.25 * pl.col("cq") / (0.25 * pl.col("sq") + pl.col("k"))).alias("ef"))
    best = x.group_by("s1").agg(pl.col("ef").max().alias("ef_best"),
                                pl.col("k").get(pl.col("ef").arg_max()).alias("k_best"),
                                pl.col("p_empty").first())
    keep = best.filter(pl.col("ef_best") > pl.col("p_empty")).select("s1", "k_best")
    return x.join(keep, on="s1").filter(pl.col("k") <= pl.col("k_best")).select("s1", "id")


def main(stage):
    pred = pl.concat([pl.read_parquet(p_path(stage, "train", j)).filter(pl.col("p") >= min(FLOORS))
                      for j in range(BUCKETS)])
    truth = pl.read_parquet(WORK / "train_gt.parquet")
    s1 = (pl.scan_parquet(WORK / "train_norm2.parquet").filter(pl.col("src") == 1)
          .select(pl.col("id").alias("s1"), "country").collect())
    base = json.load(open(WORK / f"val_stage{stage}{TAG}.json"))
    print(f"stage {stage} global threshold: F={base['f05']:.5f} at t={base['t']}", flush=True)
    res = {}
    with timer("grid"):
        for f in FLOORS + [base["t"]]:
            pa = assigned(pred, f)
            for g in GAMMAS:
                res[(f, g)] = f05(s1, decode(pa, g), truth)
                print(f"  floor {f} gamma {g}: F={res[(f, g)]:.5f}", flush=True)
    (bf, bg), bF = max(res.items(), key=lambda kv: kv[1])
    d = bF - base["f05"]
    print(f"BEST expected-F decode: floor {bf} gamma {bg} F={bF:.5f}  (global threshold {base['f05']:.5f}, "
          f"delta {d:+.5f})", flush=True)
    print(">>> VERDICT: " + ("SUBMIT the _ef file (better on train OOF)" if d > 0.0005 else
                             "keep the global-threshold file (_ef is not better)"), flush=True)
    json.dump({"floor": bf, "gamma": bg, "f05": bF, "global_f05": base["f05"],
               "grid": {f"{k[0]}_{k[1]}": v for k, v in res.items()}}, open(WORK / f"val_decode{stage}{TAG}.json", "w"))
    del pred
    with timer("write test"):
        out = OUT.parent / (OUT.name + "_ef")
        out.mkdir(parents=True, exist_ok=True)
        tp = pl.concat([pl.read_parquet(p_path(stage, "test", j)) for j in range(BUCKETS)])
        s1_all = (pl.scan_parquet(WORK / "test_norm2.parquet").filter(pl.col("src") == 1)
                  .select(pl.col("id").alias("s1")).sort("s1").collect())
        m = decode(assigned(tp.filter(pl.col("p") >= bf), bf), bg)
        to_tsv(s1_all, m, "matched_entity_ids", out / "matching_results.tsv")
        to_tsv(s1_all, tp.select("s1", "id"), "candidate_entity_ids", out / "candidate_pairs.tsv")
        print("matched pairs", m.height, "S1 with >=1 match", m["s1"].n_unique(), "of", s1_all.height, "->", out)


if __name__ == "__main__":
    main(int(sys.argv[1]) if len(sys.argv) > 1 else 2)

In [ ]:
%%writefile /kaggle/working/src/s10_v3.py
"""Blocking v3: key candidates (cand2) + embedding-kNN candidates (knn@K from e1_emb.py).
Measured on train: recall 0.9503 (keys) -> 0.9784 (keys + kNN@20), +12.8 pairs/S1.
  cos   : embedding cosine for every pair of the union -> embcos3_{split}_{j}.parquet [s1, id, emb_cos]
          (pairs sorted by embedding row, so the memmap is read nearly sequentially; S1 rows held in RAM)
  build : feat3_{split}_{j} = feat2 rows (key pairs, reused as-is) + s3 features of the NEW pairs only,
          + emb_cos, erk (kNN rank, null if not a kNN pair), in_keys, in_knn, and per-S1 windows recomputed
          over the union. Processed in S1 quarters (windows are per S1, so quarters are exact).
Then run s4/s5/s6/s8/s9 with BER_FEAT=feat3 BER_TAG=_v3 (v2 files are never overwritten).
usage: s10_v3.py cos <split> | s10_v3.py build <split> [bucket]"""
import os
import sys

import numpy as np
import polars as pl
import torch

from common import WORK, done, timer
from s2_block import BUCKETS, cand_path
import s3_features as s3

K3 = int(os.environ.get("BER_KNN_K3", 20))
QUART = 4
DIM = 384
DEV = "cuda" if torch.cuda.is_available() else "cpu"


def knn_path(split, j):
    return WORK / f"knn_{split}_{j}.parquet"


def embcos3_path(split, j):
    return WORK / f"embcos3_{split}_{j}.parquet"


def feat2_path(split, j):
    return WORK / f"{split}_feat2_{j}.parquet"


def feat3_path(split, j):
    return WORK / f"{split}_feat3_{j}.parquet"


def knnft_path(split, j):
    return WORK / f"knnft_{split}_{j}.parquet"


def knn_pairs(split, j):
    """[s1, id] of kNN@K3, plus kNN@K3 of the fine-tuned e5 (knnft_*, s17) when present. The fine-tuned model only
    adds candidates; every feature (emb_cos, erk, windows) still comes from the original e5, so train/test stay alike."""
    k = [pl.read_parquet(knn_path(split, j)).filter(pl.col("erk") <= K3).select("s1", "id")]
    if knnft_path(split, j).exists():
        k.append(pl.read_parquet(knnft_path(split, j)).filter(pl.col("erk") <= K3).select("s1", "id"))
    return pl.concat(k).unique()


def union_pairs(split, j):
    """[s1, id] of cand2 ∪ kNN@K3 (∪ fine-tuned kNN@K3) for bucket j."""
    c = pl.read_parquet(cand_path(split, j), columns=["s1", "id"])
    return pl.concat([c, knn_pairs(split, j)]).unique()


NA = 3   # anchors per S1 for the cluster-support features


def cos(split):
    """emb_cos = cosine(S1, candidate) for every union pair, plus CLUSTER SUPPORT: in train an S1 usually has 2-5
    duplicates in S2 and in S3, and true duplicates resemble each other. Anchors = each S1's NA candidates with the
    highest emb_cos; for every candidate x: sup_max / sup_mean = max / mean cosine(x, anchor) over the S1's anchors
    other than x itself, sup_n90 = how many of them have cosine > 0.9. No learned scores are used (train/test consistent)."""
    ids = np.load(WORK / f"{split}_emb_ids.npy")
    E = np.memmap(WORK / f"{split}_emb.f16", dtype=np.float16, mode="r", shape=(len(ids), DIM))
    row = pl.DataFrame({"id": ids}).with_row_index("row")
    for j in range(BUCKETS):
        if done(embcos3_path(split, j)):
            continue
        with timer(f"cos3 {split} {j}"):
            u = (union_pairs(split, j).join(row.rename({"id": "s1", "row": "r1"}), on="s1")
                 .join(row.rename({"row": "r2"}), on="id").sort("r2"))
            s1_rows = np.unique(u["r1"].to_numpy())
            S = torch.from_numpy(np.ascontiguousarray(E[s1_rows])).to(DEV).float()   # this bucket's S1 embeddings
            loc = np.searchsorted(s1_rows, u["r1"].to_numpy()).astype(np.int64)
            r2 = u["r2"].to_numpy().astype(np.int64)   # polars row index is UInt32; torch needs int64
            out = np.empty(u.height, dtype=np.float32)
            for i in range(0, u.height, 500_000):
                b = torch.from_numpy(np.ascontiguousarray(E[r2[i:i + 500_000]])).to(DEV).float()  # sorted rows
                a = S[torch.from_numpy(loc[i:i + 500_000]).to(DEV)]
                out[i:i + 500_000] = (a * b).sum(1).cpu().numpy()
            del S
            u = u.with_columns(pl.Series("emb_cos", out), pl.Series("loc", loc))
            # anchors: top-NA candidates per S1 by emb_cos -> A_idx[s1 local index, k] = index into Aemb (-1 = none)
            anc = (u.select("loc", "r2", "emb_cos").sort(["loc", "emb_cos"], descending=[False, True])
                   .group_by("loc", maintain_order=True).head(NA)
                   .with_columns(pl.int_range(pl.len()).over("loc").alias("k")))
            arows = np.unique(anc["r2"].to_numpy())
            Aemb = torch.from_numpy(np.ascontiguousarray(E[arows])).to(DEV).float()
            A_idx = np.full((len(s1_rows), NA), -1, dtype=np.int64)
            A_row = np.full((len(s1_rows), NA), -1, dtype=np.int64)
            A_idx[anc["loc"].to_numpy(), anc["k"].to_numpy()] = np.searchsorted(arows, anc["r2"].to_numpy())
            A_row[anc["loc"].to_numpy(), anc["k"].to_numpy()] = anc["r2"].to_numpy().astype(np.int64)
            A_idx_t, A_row_t = torch.from_numpy(A_idx).to(DEV), torch.from_numpy(A_row).to(DEV)
            smax = np.empty(u.height, dtype=np.float32); smean = np.empty(u.height, dtype=np.float32)
            sn90 = np.empty(u.height, dtype=np.int8)
            CH = 250_000
            for i in range(0, u.height, CH):
                bx = torch.from_numpy(np.ascontiguousarray(E[r2[i:i + CH]])).to(DEV).float()
                lt = torch.from_numpy(loc[i:i + CH]).to(DEV)
                ai, ar = A_idx_t[lt], A_row_t[lt]                                  # [n, NA]
                sims = (Aemb[ai.clamp(min=0)] * bx[:, None, :]).sum(-1)            # [n, NA]
                valid = (ai >= 0) & (ar != torch.from_numpy(r2[i:i + CH]).to(DEV)[:, None])
                nv = valid.sum(1)
                smax[i:i + CH] = torch.where(nv > 0, sims.masked_fill(~valid, -2.0).max(1).values,
                                             torch.full_like(nv, float("nan"), dtype=torch.float32)).cpu().numpy()
                smean[i:i + CH] = torch.where(nv > 0, (sims * valid).sum(1) / nv.clamp(min=1),
                                              torch.full_like(nv, float("nan"), dtype=torch.float32)).cpu().numpy()
                sn90[i:i + CH] = ((sims > 0.9) & valid).sum(1).to(torch.int8).cpu().numpy()
            del Aemb
            tmp = embcos3_path(split, j).with_suffix(".tmp")
            (u.select("s1", "id", "emb_cos")
             .with_columns(pl.Series("sup_max", smax), pl.Series("sup_mean", smean), pl.Series("sup_n90", sn90))
             .write_parquet(tmp))
            os.replace(tmp, embcos3_path(split, j))
            del u
            torch.cuda.empty_cache()   # long runs fragment GPU memory (OOM seen on a T4 in cos test)


def new_pairs(split, j, c2):
    """kNN pairs not already in cand2, shaped like cand2 rows (no shared keys: nk*=0, sc=0, rk=99)."""
    k = knn_pairs(split, j).join(c2.select("s1", "id"), on=["s1", "id"], how="anti")
    nraw = c2.group_by("s1").agg(pl.col("ncand_raw").first())
    nk = [c for c in c2.columns if c.startswith("nk")]
    new = (k.join(nraw, on="s1", how="left")
           .with_columns(*[pl.lit(0).alias(c) for c in nk], pl.lit(0).alias("sc"), pl.lit(99).alias("rk"),
                         pl.col("ncand_raw").fill_null(0)))
    return new.select(c2.columns).cast(dict(c2.schema))


def windows(df):
    """Per-S1 relative features over the whole union (the feat2 values only saw key pairs)."""
    return df.with_columns(
        (pl.col("pair_sum").max().over("s1") - pl.col("pair_sum")).alias("gap_s1_best"),
        pl.col("pair_sum").rank("min", descending=True).over("s1").cast(pl.Int16).alias("rk_s1_ps"),
        (pl.col("pair_sum2").max().over("s1") - pl.col("pair_sum2")).alias("gap_s1_best2"),
        pl.col("emb_cos").rank("min", descending=True).over("s1").cast(pl.Int16).alias("emb_rk"),
        (pl.col("emb_cos").max().over("s1") - pl.col("emb_cos")).alias("emb_gap"),
        pl.len().over("s1").cast(pl.Int16).alias("ncand3"),
    )


def build(split, buckets=range(BUCKETS)):
    idf = None
    for j in buckets:
        if done(feat3_path(split, j)):
            continue
        with timer(f"feat3 {split} {j}"):
            c2 = pl.read_parquet(cand_path(split, j))
            new = new_pairs(split, j, c2)
            del c2
            if idf is None:
                idf = {col: pl.read_parquet(s3.idf_path(split, col)).with_columns(pl.col("country").cast(pl.Utf8))
                       for col in ("ch", "ah")}
            n = s3.load_records(split, pl.concat([new["s1"], new["id"]]).unique().to_frame("id"))
            gt = None
            if split == "train":
                gt = (pl.scan_parquet(WORK / "train_gt.parquet").filter(pl.col("s1") % BUCKETS == j)
                      .with_columns(pl.lit(1, pl.Int8).alias("y")).collect())
            knn = (pl.read_parquet(knn_path(split, j)).filter(pl.col("erk") <= K3)
                   .select("s1", "id", pl.col("erk").cast(pl.Int16)))
            ec = pl.read_parquet(embcos3_path(split, j))
            print(f"  new pairs {new.height:,} ({s3.rss()})", flush=True)
            parts = []
            for q in range(QUART):
                piece = (pl.col("s1") // BUCKETS) % QUART == q
                old = pl.read_parquet(feat2_path(split, j)).filter(piece)
                nq = new.filter(piece)
                fn = s3.finish_chunk(s3.features_chunk(nq, n, idf), gt) if nq.height else old.head(0)
                u = pl.concat([old.with_columns(pl.lit(1, pl.Int8).alias("in_keys")),
                               fn.select(old.columns).with_columns(pl.lit(0, pl.Int8).alias("in_keys"))],
                              how="vertical_relaxed")
                u = (u.join(ec, on=["s1", "id"], how="left").join(knn, on=["s1", "id"], how="left")
                     .with_columns(pl.col("erk").is_not_null().cast(pl.Int8).alias("in_knn")))
                pf = WORK / f"_tmp3_{split}_{j}_{q}.parquet"
                windows(u).write_parquet(pf)
                parts.append(pf)
                del old, nq, fn, u
                print(f"  quarter {q + 1}/{QUART} {s3.rss()}", flush=True)
            del n, new, knn, ec
            tmp = feat3_path(split, j).with_suffix(".tmp")
            pl.scan_parquet(parts).sink_parquet(tmp)
            os.replace(tmp, feat3_path(split, j))
            for pf in parts:
                pf.unlink()


def report():
    """Recall of feat3 train vs ground truth (should match the kNN@K3 union recall, ~0.978 for K=20)."""
    gt = pl.read_parquet(WORK / "train_gt.parquet")
    hit = tot = rows = 0
    for j in range(BUCKETS):
        f = pl.read_parquet(feat3_path("train", j), columns=["s1", "id", "y"])
        g = gt.filter(pl.col("s1") % BUCKETS == j)
        hit += int(f["y"].sum()); tot += g.height; rows += f.height
    print(f"feat3 train: {rows:,} pairs, positives {hit:,} / {tot:,} = recall {hit / tot:.4f}", flush=True)


if __name__ == "__main__":
    step, split = sys.argv[1], (sys.argv[2] if len(sys.argv) > 2 else "train")
    if step == "cos":
        cos(split)
    elif step == "build":
        build(split, [int(sys.argv[3])] if len(sys.argv) > 3 else range(BUCKETS))
    elif step == "report":
        report()

In [ ]:
%%writefile /kaggle/working/src/s11_tune.py
"""Threshold tuning under test-like decoy density.
Test has ~18% fewer S1 per S2/S3 record than train (train 4.67 S2+S3 per S1, test 5.5-5.8), i.e. more look-alike records
whose true entity is not in S1 (decoys). Simulate that on train OOF predictions: drop DROP of the S1 entities (their true
S2/S3 records stay in the pool as decoys), re-run the global assignment, score macro F0.5 on the remaining S1.
Prints the curve per seed, picks the best threshold, and (with --write) writes the test submission at that threshold.
usage: s11_tune.py <stage> [--write]    (reads p{stage}{TAG}_train/test; LB check: v3 s1 t=0.625 -> 0.961, t=0.80 -> 0.963)"""
import json
import os
import sys

import numpy as np
import polars as pl

from common import OUT, TAG, WORK, f05, timer
from s2_block import BUCKETS
from s4_train import assign, p_path

DROP = float(os.environ.get("BER_DROP", 0.18))
SEEDS = (1, 2)
GRID = [round(x, 3) for x in np.arange(0.5, 0.981, 0.025)]


def main(stage, write):
    pred = pl.concat([pl.read_parquet(p_path(stage, "train", j), columns=["s1", "id", "p"]).filter(pl.col("p") >= GRID[0])
                      for j in range(BUCKETS)])
    truth = pl.read_parquet(WORK / "train_gt.parquet")
    s1 = (pl.scan_parquet(WORK / "train_norm2.parquet").filter(pl.col("src") == 1)
          .select(pl.col("id").alias("s1")).collect())
    base = json.load(open(WORK / f"val_stage{stage}{TAG}.json"))
    print(f"stage {stage}{TAG}: normal OOF best t={base['t']} F={base['f05']:.4f}", flush=True)
    curves = []
    for seed in SEEDS:
        with timer(f"seed {seed}"):
            keep = s1.filter(pl.struct("s1").hash(seed=seed) % 1000 >= int(DROP * 1000))
            pk = pred.join(keep, on="s1", how="semi")        # dropped S1 vanish; their records stay as decoys
            tk = truth.join(keep, on="s1", how="semi")
            curve = {t: f05(keep, assign(pk, t), tk) for t in GRID}
            curves.append(curve)
            b = max(curve, key=curve.get)
            print(f"  seed {seed}: kept {keep.height:,} S1 | best t={b} F={curve[b]:.4f} | "
                  + " ".join(f"{t}:{curve[t]:.4f}" for t in GRID[::2]), flush=True)
    mean = {t: float(np.mean([c[t] for c in curves])) for t in GRID}
    bt = max(mean, key=mean.get)
    print(f"DECOY-SIM best t={bt} (mean F {mean[bt]:.4f}); normal-OOF t={base['t']}", flush=True)
    json.dump({"t": bt, "drop": DROP, "curve": mean, "normal_t": base["t"]},
              open(WORK / f"val_decoy{stage}{TAG}.json", "w"))
    if write:
        from s6_write import to_tsv
        out = OUT.parent / (OUT.name + f"_t{int(round(bt * 100))}")
        out.mkdir(parents=True, exist_ok=True)
        p = pl.concat([pl.read_parquet(p_path(stage, "test", j), columns=["s1", "id", "p"]).filter(pl.col("p") >= bt)
                       for j in range(BUCKETS)])
        s1_all = (pl.scan_parquet(WORK / "test_norm2.parquet").filter(pl.col("src") == 1)
                  .select(pl.col("id").alias("s1")).sort("s1").collect())
        m = assign(p, bt)
        to_tsv(s1_all, m, "matched_entity_ids", out / "matching_results.tsv")
        print(f"wrote {out}/matching_results.tsv: t={bt}, matched pairs {m.height:,}", flush=True)


if __name__ == "__main__":
    main(int(sys.argv[1]), "--write" in sys.argv)

In [ ]:
%%writefile /kaggle/working/src/s13_errors.py
"""Error budget of a stage's OOF decisions (train). How much macro F0.5 would we gain by removing each error type alone?
  FP_single : predicted matches on S1 that have NO true match (singletons)
  FP_owned  : predicted id truly belongs to ANOTHER S1 (wrong assignment)
  FP_decoy  : predicted id matches nothing at all (look-alike decoy)
  FP_other  : the rest of the false pairs on matched S1
  FN_cand   : true pair in the candidates but not predicted (low p, or the id was assigned elsewhere)
  FN_block  : true pair not in the candidates at all
usage: s13_errors.py <stage>   (BER_TAG=_v3 for v3 files; needs train_gt, train_norm2, p{stage}{TAG}_train_*, knn/cand for FN_block)"""
import json
import sys

import polars as pl

from common import TAG, WORK, f05_scores
from s2_block import BUCKETS
from s4_train import assign, p_path


def main(stage):
    t = json.load(open(WORK / f"val_stage{stage}{TAG}.json"))["t"]
    s1 = (pl.scan_parquet(WORK / "train_norm2.parquet").filter(pl.col("src") == 1)
          .select(pl.col("id").alias("s1"), "country").collect())
    truth = pl.read_parquet(WORK / "train_gt.parquet")
    p = pl.concat([pl.read_parquet(p_path(stage, "train", j), columns=["s1", "id", "p"]) for j in range(BUCKETS)])
    pred = assign(p.filter(pl.col("p") >= t), t)
    cand = p.select("s1", "id")
    base = f05_scores(s1, pred, truth)["f"].mean()
    tp = pred.join(truth, on=["s1", "id"], how="semi")
    fp = pred.join(truth, on=["s1", "id"], how="anti")
    matched_s1 = truth.select("s1").unique()
    owner = truth.rename({"s1": "true_s1"})
    fp = fp.join(owner, on="id", how="left").with_columns(
        pl.when(~pl.col("s1").is_in(matched_s1["s1"].implode())).then(pl.lit("FP_single"))
        .when(pl.col("true_s1").is_not_null()).then(pl.lit("FP_owned"))
        .otherwise(pl.lit("FP_decoy")).alias("kind"))
    fn = truth.join(pred, on=["s1", "id"], how="anti")
    fn = pl.concat([fn.join(cand, on=["s1", "id"], how="semi").with_columns(pl.lit("FN_cand").alias("kind")),
                    fn.join(cand, on=["s1", "id"], how="anti").with_columns(pl.lit("FN_block").alias("kind"))])
    print(f"stage {stage}{TAG} t={t}: OOF macro F0.5 {base:.4f} | pairs: TP {tp.height:,} FP {fp.height:,} FN {fn.height:,}", flush=True)
    print("  pair precision %.4f  pair recall %.4f" % (tp.height / max(pred.height, 1), tp.height / truth.height), flush=True)
    print("  counts:", dict(sorted(fp.group_by("kind").len().rows() + fn.group_by("kind").len().rows())), flush=True)
    print("  macro-F0.5 gain if ONLY this error type were fixed:", flush=True)
    for k in ("FP_single", "FP_owned", "FP_decoy"):
        bad = fp.filter(pl.col("kind") == k).select("s1", "id")
        f = f05_scores(s1, pred.join(bad, on=["s1", "id"], how="anti"), truth)["f"].mean()
        print(f"    {k:10s} n={bad.height:>9,}  +{f - base:.4f}", flush=True)
    for k in ("FN_cand", "FN_block"):
        add = fn.filter(pl.col("kind") == k).select("s1", "id")
        f = f05_scores(s1, pl.concat([pred, add]), truth)["f"].mean()
        print(f"    {k:10s} n={add.height:>9,}  +{f - base:.4f}", flush=True)
    sc = f05_scores(s1, pred, truth).join(s1, on="s1")
    print("  by country:", {r[0]: round(r[1], 4) for r in sc.group_by("country").agg(pl.col("f").mean()).rows()}, flush=True)
    print("  singletons: share of S1 %.3f, F on them %.4f | matched S1: F %.4f" % (
        (sc["nt"] == 0).mean(), sc.filter(pl.col("nt") == 0)["f"].mean(), sc.filter(pl.col("nt") > 0)["f"].mean()), flush=True)


if __name__ == "__main__":
    main(int(sys.argv[1]) if len(sys.argv) > 1 else 1)

In [ ]:
%%writefile /kaggle/working/src/s14_addr.py
"""Add-on address features for the house-number noise seen in the error budget: among rejected true pairs with an address,
72% have a different first house number and 45% of those are one digit edit away (1161 vs 161, 85 vs 15).
Per pair of the v3 union (cand2 + kNN@20, the same pairs as feat3):
  street_tset / street_ratio : address similarity with every token containing a digit removed
  num_best_lev               : smallest Levenshtein distance between any of the first 3 numbers on each side (9 = none)
  num_contain                : first numbers differ but one contains the other (161 in 1161)
  num_reldiff                : |a - b| / max(a, b) of the first numbers (NaN if missing)
Out: addr4_{split}_{j}.parquet [s1, id, ...] (model-independent, so untagged). Joined by s4_train as an extra (stages 3-5).
usage: s14_addr.py <split>"""
import os
import sys

import numpy as np
import polars as pl
from rapidfuzz import fuzz
from rapidfuzz.distance import Levenshtein
from rapidfuzz.process import cpdist

from common import WORK, done, timer
from s2_block import BUCKETS
import s3_features as s3
from s10_v3 import union_pairs   # same pairs as feat3 (incl. fine-tuned kNN pairs when present)

SUB = 8


def addr_path(split, j):
    return WORK / f"addr4_{split}_{j}.parquet"




def chunk_feats(x):
    street = lambda c: pl.col(c).str.replace_all(r"\S*\d\S*", " ").str.replace_all(r"\s+", " ").str.strip_chars()
    x = x.with_columns(street("addr_n").alias("st1"), street("addr_n_2").alias("st2"))
    st1, st2 = x["st1"].to_list(), x["st2"].to_list()
    feats = {"street_tset": cpdist(st1, st2, scorer=fuzz.token_set_ratio, workers=-1, dtype=np.uint8),
             "street_ratio": cpdist(st1, st2, scorer=fuzz.ratio, workers=-1, dtype=np.uint8)}
    best = np.full(x.height, 9, dtype=np.int8)
    for a in range(3):
        na = x["nums"].list.get(a, null_on_oob=True)
        for b in range(3):
            nb = x["nums_2"].list.get(b, null_on_oob=True)
            ok = (na.is_not_null() & nb.is_not_null()).to_numpy()
            if not ok.any():
                continue
            d = cpdist(na.fill_null("").to_list(), nb.fill_null("").to_list(), scorer=Levenshtein.distance, workers=-1,
                       dtype=np.int32)
            best = np.where(ok, np.minimum(best, d), best).astype(np.int8)
    feats["num_best_lev"] = best
    f1, f2 = pl.col("nums").list.first(), pl.col("nums_2").list.first()
    i1, i2 = f1.str.slice(0, 9).cast(pl.Int64, strict=False), f2.str.slice(0, 9).cast(pl.Int64, strict=False)
    out = x.select(
        "s1", "id",
        ((f1 != f2) & (f1.str.contains(f2, literal=True) | f2.str.contains(f1, literal=True))).fill_null(False).alias("num_contain"),
        ((i1 - i2).abs() / pl.max_horizontal(i1, i2).clip(lower_bound=1)).cast(pl.Float32).alias("num_reldiff"),
    )
    return out.with_columns(*[pl.Series(k, v) for k, v in feats.items()])


def build(split):
    for j in range(BUCKETS):
        if done(addr_path(split, j)):
            continue
        with timer(f"addr4 {split} {j}"):
            u = union_pairs(split, j)
            n = s3.load_records(split, pl.concat([u["s1"], u["id"]]).unique().to_frame("id")).select("id", "addr_n", "nums")
            parts = []
            for s in range(SUB):
                c = u.filter((pl.col("s1") // BUCKETS) % SUB == s)
                x = (c.join(n.rename({"id": "s1"}), on="s1").join(n, on="id", suffix="_2"))
                parts.append(chunk_feats(x))
                del c, x
            tmp = addr_path(split, j).with_suffix(".tmp")
            pl.concat(parts).write_parquet(tmp)
            os.replace(tmp, addr_path(split, j))
            del u, n, parts


if __name__ == "__main__":
    build(sys.argv[1])

In [ ]:
%%writefile /kaggle/working/src/s15_blend.py
"""Blend + write: average the probabilities of one or more stages (same pairs), tune the threshold on train OOF (macro F0.5,
global assignment), and write two test submissions:
  <OUT>/<name>/matching_results.tsv        at the OOF-best threshold
  <OUT>/<name>_fr90/matching_results.tsv   at t=0.80 (France 0.90)  - the stricter setting that won on the public LB
  <OUT>/<name>_t80/matching_results.tsv    at t=0.80 everywhere (France too): pairs with _fr90 to measure France's threshold
usage: s15_blend.py <name> <stage> [<stage> ...]      e.g.  s15_blend.py v3s5full 5    |   s15_blend.py blend56 5 6
       s15_blend.py write <name> <t> <t_france> <stage> [<stage> ...]   write one file from existing test p (no tuning)"""
import json
import os
import sys

import numpy as np
import polars as pl

from common import OUT, WORK, f05, timer
from s2_block import BUCKETS
from s4_train import assign, p_path
from s6_write import to_tsv

LOW = 0.4
GRID = [round(x, 3) for x in np.arange(0.5, 0.951, 0.025)]


def blended(stages, split, j, low=None):
    ps = [pl.read_parquet(p_path(st, split, j), columns=["s1", "id", "p"]).rename({"p": f"p{k}"}) for k, st in enumerate(stages)]
    b = ps[0]
    for q in ps[1:]:
        b = b.join(q, on=["s1", "id"], how="inner")
    b = b.select("s1", "id", pl.mean_horizontal([f"p{k}" for k in range(len(stages))]).alias("p"))
    return b if low is None else b.filter(pl.col("p") >= low)


def write(name, stages, rules):
    """rules: [(sub-folder name, None = OOF-best t (needs bt) or (t, t_france))]."""
    tp = pl.concat([blended(stages, "test", j, LOW) for j in range(BUCKETS)])
    s1_all = (pl.scan_parquet(WORK / "test_norm2.parquet").filter(pl.col("src") == 1)
              .select(pl.col("id").alias("s1")).sort("s1").collect())
    cty = pl.read_parquet(WORK / "test_records.parquet", columns=["id", "country"]).rename({"id": "s1"})
    for sub, rule in rules:
        if isinstance(rule, float):
            m = assign(tp, rule)
        else:
            q = tp.join(cty, on="s1").filter(pl.when(pl.col("country") == "France").then(pl.col("p") >= rule[1])
                                             .otherwise(pl.col("p") >= rule[0]))
            m = q.sort("p", descending=True).unique("id", keep="first").select("s1", "id")
        d = OUT / sub
        d.mkdir(parents=True, exist_ok=True)
        to_tsv(s1_all, m, "matched_entity_ids", d / "matching_results.tsv")
        print(f"  wrote {d}/matching_results.tsv  matched pairs {m.height:,}", flush=True)
    return tp, s1_all


def main(name, stages):
    with timer(f"tune {name} {stages}"):
        pred = pl.concat([blended(stages, "train", j, LOW) for j in range(BUCKETS)])
        truth = pl.read_parquet(WORK / "train_gt.parquet")
        s1 = (pl.scan_parquet(WORK / "train_norm2.parquet").filter(pl.col("src") == 1)
              .select(pl.col("id").alias("s1"), "country").collect())
        curve = {t: f05(s1, assign(pred, t), truth) for t in GRID}
        bt = max(curve, key=curve.get)
        by = {c: f05(s1.filter(pl.col("country") == c), assign(pred, bt).join(s1.filter(pl.col("country") == c), on="s1", how="semi"),
                     truth.join(s1.filter(pl.col("country") == c), on="s1", how="semi")) for c in s1["country"].unique().to_list()}
        print(f"{name} {stages}: OOF best t={bt} F0.5={curve[bt]:.5f} | at 0.80: {curve[0.8]:.5f} | by country "
              + str({k: round(v, 4) for k, v in by.items()}), flush=True)
        json.dump({"stages": stages, "t": bt, "f05": curve[bt], "curve": curve, "by_country": by},
                  open(WORK / f"val_blend_{name}.json", "w"))
    with timer(f"write {name}"):
        tp, s1_all = write(name, stages, [(name, float(bt)), (name + "_fr90", (0.80, 0.90)), (name + "_t80", (0.80, 0.80))])
        cand = OUT / "candidates" / "candidate_pairs.tsv"
        if not cand.exists():
            cand.parent.mkdir(parents=True, exist_ok=True)
            allp = pl.concat([pl.read_parquet(p_path(stages[0], "test", j), columns=["s1", "id"]) for j in range(BUCKETS)])
            to_tsv(s1_all, allp, "candidate_entity_ids", cand)
            print(f"  wrote {cand}", flush=True)


if __name__ == "__main__":
    if sys.argv[1] == "write":
        nm, t, tf = sys.argv[2], float(sys.argv[3]), float(sys.argv[4])
        with timer(f"write {nm} t={t} France={tf}"):
            write(nm, [int(x) for x in sys.argv[5:]], [(nm, (t, tf))])
    else:
        main(sys.argv[1], [int(x) for x in sys.argv[2:]])

In [ ]:
%%writefile /kaggle/working/src/s16_idcomp.py
"""Per-record competition features, computed from similarities only (no model scores -> identical behaviour on train/test).
Half of all S1 names are shared by several S1 businesses, so for one S2/S3 record the model must pick WHICH same-named S1
has the best address. Each pair is otherwise scored in isolation. Here, for every pair (s1, id) of the v3 union:
  idc_n        : number of S1 candidates of this record (id)
  idc_rk_ps / idc_gap_ps   : rank of pair_sum (name+address fuzzy score) among the record's candidates / gap to the best rival
  idc_rk_ad / idc_gap_ad   : same for addr_tset
  idc_rk_em / idc_gap_em   : same for emb_cos
  idc_n_name90 : how many of the record's candidates have core_tset >= 90 (near-identical names = ambiguity)
  s1c_n_name90 : how many candidates of this S1 have core_tset >= 90
Needs all 8 feat3 buckets of a split (a record's candidates span buckets). Out: idc_{split}_{j}.parquet.
usage: s16_idcomp.py <split>"""
import os
import sys

import polars as pl

from common import WORK, done, timer
from s2_block import BUCKETS

COLS = ["s1", "id", "pair_sum", "addr_tset", "emb_cos", "core_tset"]


def idc_path(split, j):
    return WORK / f"idc_{split}_{j}.parquet"


def feat3(split, j):
    return WORK / f"{split}_feat3_{j}.parquet"


def build(split):
    if all(done(idc_path(split, j)) for j in range(BUCKETS)):
        return
    with timer(f"idcomp {split}"):
        x = pl.concat([pl.read_parquet(feat3(split, j), columns=COLS) for j in range(BUCKETS)])
        g = (x.group_by("id").agg(pl.len().cast(pl.Int16).alias("idc_n"),
                                  pl.col("pair_sum").max().alias("mx_ps"), pl.col("addr_tset").max().alias("mx_ad"),
                                  pl.col("emb_cos").max().alias("mx_em"),
                                  (pl.col("core_tset") >= 90).sum().cast(pl.Int16).alias("idc_n_name90")))
        s = x.group_by("s1").agg((pl.col("core_tset") >= 90).sum().cast(pl.Int16).alias("s1c_n_name90"))
        x = x.join(g, on="id", how="left").join(s, on="s1", how="left")
        x = x.with_columns(
            pl.col("pair_sum").rank("min", descending=True).over("id").cast(pl.Int16).alias("idc_rk_ps"),
            pl.col("addr_tset").rank("min", descending=True).over("id").cast(pl.Int16).alias("idc_rk_ad"),
            pl.col("emb_cos").rank("min", descending=True).over("id").cast(pl.Int16).alias("idc_rk_em"),
            (pl.col("mx_ps") - pl.col("pair_sum")).cast(pl.Int16).alias("idc_gap_ps"),
            (pl.col("mx_ad").cast(pl.Int16) - pl.col("addr_tset").cast(pl.Int16)).alias("idc_gap_ad"),
            (pl.col("mx_em") - pl.col("emb_cos")).cast(pl.Float32).alias("idc_gap_em"),
        ).select("s1", "id", "idc_n", "idc_rk_ps", "idc_gap_ps", "idc_rk_ad", "idc_gap_ad", "idc_rk_em", "idc_gap_em",
                 "idc_n_name90", "s1c_n_name90")
        for j in range(BUCKETS):
            tmp = idc_path(split, j).with_suffix(".tmp")
            x.filter(pl.col("s1") % BUCKETS == j).write_parquet(tmp)
            os.replace(tmp, idc_path(split, j))
        del x


if __name__ == "__main__":
    build(sys.argv[1])

In [ ]:
%%writefile /kaggle/working/src/e1_emb.py
"""Parallel path E: multilingual-e5-small embeddings for (1) kNN blocking and (2) a cosine feature.
  encode  : "query: <name> | <address>" for every record -> {split}_emb.f16 (memmap [N, 384] float16, L2-normalised)
            row order = {split}_records.parquet row order (ids in {split}_emb_ids.npy)
  knn     : within each country, each S1 -> top-K S2/S3 by cosine (GPU, tiled matmul) -> knn_{split}_{j}.parquet
            [s1, id, cos, erk]   (j = S1 bucket, same bucketing as cand2)
  recall  : train only: recall of cand2, kNN, and their union vs ground truth; new pairs per S1
  cos     : cosine for every existing cand2 pair -> embcos_{split}_{j}.parquet [s1, id, emb_cos, emb_rk]
            (joined automatically by s4_train stage 3 when present)
Every step skips outputs that already exist. Nothing big is held in RAM (memmap + chunks)."""
import os
import sys
import time

import numpy as np
import polars as pl
import torch
from transformers import AutoModel, AutoTokenizer

from common import WORK, done, timer
from s2_block import BUCKETS, cand_path

MODEL = os.environ.get("BER_EMB_MODEL", "/content/drive/MyDrive/MLC'26/models/multilingual-e5-small")
K = int(os.environ.get("BER_KNN_K", 20))
DIM, MAXLEN, BS = 384, 64, 1024
QB = 1024   # S1 rows per GPU block: scores tile = QB x 1M fp16 = 2 GB
DEV = "cuda" if torch.cuda.is_available() else "cpu"


def emb_path(split):
    return WORK / f"{split}_emb.f16"


def ids_path(split):
    return WORK / f"{split}_emb_ids.npy"


def knn_path(split, j):
    return WORK / f"knn_{split}_{j}.parquet"


def embcos_path(split, j):
    return WORK / f"embcos_{split}_{j}.parquet"


def records(split):
    return pl.read_parquet(WORK / f"{split}_records.parquet", columns=["id", "src", "name", "addr", "country"])


def open_emb(split, mode="r"):
    n = len(np.load(ids_path(split)))
    return np.memmap(emb_path(split), dtype=np.float16, mode=mode, shape=(n, DIM))


NCHUNK = 8   # encode in 8 chunks; each finished chunk is saved (and copied to BER_EMB_SYNC if set)


def chunk_path(split, c):
    return WORK / f"{split}_emb_c{c}.f16"


@torch.no_grad()
def encode(split):
    """Resumable: records are encoded shortest-first in NCHUNK chunks. A finished chunk is written as
    {split}_emb_c{c}.f16 and, if BER_EMB_SYNC is set, copied there (Drive) at once, so a crash or a dead session
    loses at most one chunk. Chunks already present are skipped. Finally the chunks are assembled into
    {split}_emb.f16 in records row order (same layout as before)."""
    import shutil
    if done(emb_path(split)) and done(ids_path(split)):
        return
    sync = os.environ.get("BER_EMB_SYNC")
    with timer(f"encode {split}"):
        r = records(split)
        np.save(ids_path(split), r["id"].to_numpy())
        txt = ("query: " + r["name"] + " | " + r["addr"]).str.slice(0, 250).to_list()
        del r
        n = len(txt)
        order = np.argsort(np.fromiter((len(t) for t in txt), dtype=np.int32, count=n), kind="stable")
        bounds = np.linspace(0, n, NCHUNK + 1).astype(np.int64)
        tok = model = None
        for c in range(NCHUNK):
            if done(chunk_path(split, c)):
                print(f"  chunk {c} present, skipped", flush=True)
                continue
            if model is None:
                tok = AutoTokenizer.from_pretrained(MODEL)
                model = AutoModel.from_pretrained(MODEL).to(DEV).half().eval()
            idx_c = order[bounds[c]:bounds[c + 1]]
            tmp = chunk_path(split, c).with_suffix(".part")
            out = np.memmap(tmp, dtype=np.float16, mode="w+", shape=(len(idx_c), DIM))
            t0 = time.time()
            for k in range(0, len(idx_c), BS):
                idx = idx_c[k:k + BS]
                enc = tok([txt[i] for i in idx], padding=True, truncation=True, max_length=MAXLEN,
                          return_tensors="pt").to(DEV)
                h = model(**enc).last_hidden_state
                m = enc["attention_mask"].unsqueeze(-1).to(h.dtype)
                e = torch.nn.functional.normalize(((h * m).sum(1) / m.sum(1)).float(), dim=1)
                out[k:k + len(idx)] = e.half().cpu().numpy()
                if k % (BS * 300) == 0:
                    el = time.time() - t0
                    print(f"  chunk {c + 1}/{NCHUNK}: {k:,}/{len(idx_c):,} {(k + BS) / max(el, 1e-6):.0f}/s "
                          f"chunk eta {el / (k + BS) * (len(idx_c) - k) / 60:.0f} min", flush=True)
            out.flush()
            del out
            os.replace(tmp, chunk_path(split, c))
            if sync:
                shutil.copy(chunk_path(split, c), os.path.join(sync, chunk_path(split, c).name + ".part"))
                os.replace(os.path.join(sync, chunk_path(split, c).name + ".part"), os.path.join(sync, chunk_path(split, c).name))
            print(f"  CHUNK {c + 1}/{NCHUNK} DONE ({time.time() - t0:.0f}s){' + saved to Drive' if sync else ''}", flush=True)
        # assemble in records row order (sorted writes -> near-sequential I/O)
        tmp = WORK / f"{split}_emb.f16.part"
        E = np.memmap(tmp, dtype=np.float16, mode="w+", shape=(n, DIM))
        for c in range(NCHUNK):
            pos = order[bounds[c]:bounds[c + 1]]
            C = np.memmap(chunk_path(split, c), dtype=np.float16, mode="r", shape=(len(pos), DIM))
            srt = np.argsort(pos)
            for i in range(0, len(pos), 1_000_000):
                sl = srt[i:i + 1_000_000]
                E[pos[sl]] = C[sl]
            del C
        E.flush()
        del E
        os.replace(tmp, emb_path(split))   # only a finished file gets the final name


@torch.no_grad()
def knn(split):
    if all(done(knn_path(split, j)) for j in range(BUCKETS)):
        return
    E = open_emb(split)
    r = pl.read_parquet(WORK / f"{split}_records.parquet", columns=["id", "src", "country"]).with_row_index("row")
    res = []
    for c in r["country"].unique().to_list():
        with timer(f"knn {split} {c}"):
            q = r.filter((pl.col("country") == c) & (pl.col("src") == 1))
            d = r.filter((pl.col("country") == c) & (pl.col("src") != 1))
            if q.height == 0 or d.height == 0:
                continue
            drow, did = d["row"].to_numpy(), d["id"].to_numpy()
            # database on GPU in chunks of 1M rows (fp16: ~0.77 GB per chunk)
            D = [torch.from_numpy(np.ascontiguousarray(E[np.sort(drow[i:i + 1_000_000])])).to(DEV)
                 for i in range(0, len(drow), 1_000_000)]
            dorder = np.concatenate([np.sort(drow[i:i + 1_000_000]) for i in range(0, len(drow), 1_000_000)])
            row2id = dict(zip(drow.tolist(), did.tolist()))
            did_sorted = np.fromiter((row2id[x] for x in dorder), dtype=np.int64, count=len(dorder))
            qrow, qid = q["row"].to_numpy(), q["id"].to_numpy()
            for i in range(0, len(qrow), QB):
                Q = torch.from_numpy(np.ascontiguousarray(E[qrow[i:i + QB]])).to(DEV)
                best_s, best_i, off = [], [], 0
                for Dc in D:
                    s = Q @ Dc.T
                    v, ix = s.topk(min(K, Dc.shape[0]), dim=1)
                    best_s.append(v.float()); best_i.append(ix + off); off += Dc.shape[0]
                v, ix = torch.cat(best_s, 1).topk(K, dim=1)
                gi = torch.gather(torch.cat(best_i, 1), 1, ix).cpu().numpy()
                v = v.cpu().numpy()
                res.append(pl.DataFrame({"s1": np.repeat(qid[i:i + QB], K), "id": did_sorted[gi.ravel()],
                                         "cos": v.ravel().astype(np.float32),
                                         "erk": np.tile(np.arange(1, K + 1, dtype=np.int16), len(gi))}))
            del D
            torch.cuda.empty_cache()
    res = pl.concat(res)
    for j in range(BUCKETS):
        res.filter(pl.col("s1") % BUCKETS == j).write_parquet(knn_path(split, j))


def recall():
    """Bucket by bucket (loading all 110M candidate pairs at once crashed a 12.7 GB runtime)."""
    tot = hc = n1 = ncand = 0
    hk, hu, nu = {5: 0, 10: 0, 20: 0}, {5: 0, 10: 0, 20: 0}, {5: 0, 10: 0, 20: 0}
    for j in range(BUCKETS):
        gt = pl.read_parquet(WORK / "train_gt.parquet").filter(pl.col("s1") % BUCKETS == j)
        cand = pl.read_parquet(cand_path("train", j), columns=["s1", "id"])
        kn = pl.read_parquet(knn_path("train", j))
        n1 += cand["s1"].n_unique(); ncand += cand.height; tot += gt.height
        hc += gt.join(cand, on=["s1", "id"], how="semi").height
        for k in (5, 10, 20):
            kk = kn.filter(pl.col("erk") <= k).select("s1", "id")
            u = pl.concat([cand, kk]).unique()
            hk[k] += gt.join(kk, on=["s1", "id"], how="semi").height
            hu[k] += gt.join(u, on=["s1", "id"], how="semi").height
            nu[k] += u.height - cand.height
        del cand, kn
    print(f"cand2 recall {hc / tot:.4f}")
    for k in (5, 10, 20):
        print(f"kNN@{k}: alone {hk[k] / tot:.4f} | union {hu[k] / tot:.4f} | new pairs/S1 {nu[k] / n1:.1f}", flush=True)


def cosine(split):
    E = open_emb(split)
    row = pl.DataFrame({"id": np.load(ids_path(split))}).with_row_index("row")
    for j in range(BUCKETS):
        if done(embcos_path(split, j)):
            continue
        with timer(f"cos {split} {j}"):
            c = (pl.read_parquet(cand_path(split, j), columns=["s1", "id"])
                 .join(row.rename({"id": "s1", "row": "r1"}), on="s1").join(row.rename({"row": "r2"}), on="id"))
            out = np.empty(c.height, dtype=np.float32)
            r1, r2 = c["r1"].to_numpy(), c["r2"].to_numpy()
            for i in range(0, c.height, 1_000_000):
                a = torch.from_numpy(E[r1[i:i + 1_000_000]]).to(DEV).float()
                b = torch.from_numpy(E[r2[i:i + 1_000_000]]).to(DEV).float()
                out[i:i + 1_000_000] = (a * b).sum(1).cpu().numpy()
            (c.select("s1", "id").with_columns(pl.Series("emb_cos", out))
             .with_columns(pl.col("emb_cos").rank("min", descending=True).over("s1").cast(pl.Int16).alias("emb_rk"))
             .write_parquet(embcos_path(split, j)))


if __name__ == "__main__":
    # usage: e1_emb.py encode|knn|recall|cos <split...>
    step, splits = sys.argv[1], sys.argv[2:] or ["train", "test"]
    for s in splits:
        {"encode": encode, "knn": knn, "cos": cosine}.get(step, lambda _: None)(s)
    if step == "recall":
        recall()

In [ ]:
%%writefile /kaggle/working/src/s17_ft_retrieval.py
"""Quick validation: does fine-tuning e5-small on our own train pairs improve kNN retrieval (blocking recall)?
Plain PyTorch (no sentence-transformers). Text = "query: <name> | <address>" (same as the current kNN).
  1. held-out S1 = bucket 7 (s1 % 8 == 7), sample N_EVAL; training pairs = (S1, matched S2/S3) from the other buckets, sample N_TRAIN
  2. eval database = all true matches of the held-out S1 + N_DB random other S2/S3 records (same countries)
  3. baseline: original e5 -> within-country kNN@20 recall on the held-out true pairs
  4. fine-tune 1 epoch, in-batch negatives (MultipleNegativesRanking, symmetric, scale 20), fp16, max_len 64
  5. fine-tuned e5 -> same recall. Saves the fine-tuned model to OUT_MODEL.
env: BER_DATA_DIR (train_records/train_gt), BER_E5 (model dir), BER_FT_OUT, BER_N_TRAIN, BER_N_EVAL, BER_N_DB"""
import os
import time

import numpy as np
import polars as pl
import torch
from transformers import AutoModel, AutoTokenizer

DATA = os.environ.get("BER_DATA_DIR", "/content/work")
E5 = os.environ.get("BER_E5", "/content/drive/MyDrive/MLC'26/models/multilingual-e5-small")
OUT_MODEL = os.environ.get("BER_FT_OUT", "/content/e5_ft")
N_TRAIN = int(os.environ.get("BER_N_TRAIN", 300_000))
N_EVAL = int(os.environ.get("BER_N_EVAL", 100_000))
N_DB = int(os.environ.get("BER_N_DB", 800_000))
BS, LR, MAXLEN, SCALE, K = 128, 2e-5, 64, 20.0, 20
DEV = "cuda" if torch.cuda.is_available() else "cpu"


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


@torch.no_grad()
def encode(model, tok, texts, bs=512):
    model.eval()
    order = np.argsort([len(t) for t in texts])
    out = torch.empty((len(texts), model.config.hidden_size), dtype=torch.float16)
    for k in range(0, len(texts), bs):
        idx = order[k:k + bs]
        enc = tok([texts[i] for i in idx], padding=True, truncation=True, max_length=MAXLEN, return_tensors="pt").to(DEV)
        with torch.autocast(DEV, dtype=torch.float16, enabled=DEV == "cuda"):
            h = model(**enc).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1).to(h.dtype)
        e = torch.nn.functional.normalize(((h * m).sum(1) / m.sum(1)).float(), dim=1)
        out[torch.from_numpy(idx)] = e.half().cpu()
    return out


def recall_at(model, tok, q, db, truth):
    """q/db: frames [id, country, t]; truth [s1, id]. Within-country kNN; returns recall@5/10/20."""
    Eq, Ed = encode(model, tok, q["t"].to_list()), encode(model, tok, db["t"].to_list())
    hits = {5: 0, 10: 0, 20: 0}
    tset = set(zip(truth["s1"].to_list(), truth["id"].to_list()))
    for c in q["country"].unique().to_list():
        qi = np.where((q["country"] == c).to_numpy())[0]
        di = np.where((db["country"] == c).to_numpy())[0]
        if len(qi) == 0 or len(di) == 0:
            continue
        D = Ed[di].to(DEV)
        qid, did = q["id"].to_numpy()[qi], db["id"].to_numpy()[di]
        for s in range(0, len(qi), 2048):
            Q = Eq[qi[s:s + 2048]].to(DEV)
            top = (Q @ D.T).topk(min(K, len(di)), dim=1).indices.cpu().numpy()
            for r, row in enumerate(top):
                s1 = qid[s + r]
                for k in (5, 10, 20):
                    hits[k] += sum((s1, did[x]) in tset for x in row[:k])
    return {k: round(v / len(tset), 4) for k, v in hits.items()}


def finetune(model, tok, a_txt, b_txt, seed=3):
    """1 epoch, in-batch negatives (symmetric MultipleNegativesRanking, scale SCALE), fp16, linear warmup/decay. In place."""
    perm = np.random.default_rng(seed).permutation(len(a_txt))
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = len(perm) // BS
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / max(1, int(0.05 * steps))) * max(0.0, 1 - s / steps))
    scaler = torch.amp.GradScaler(enabled=DEV == "cuda")
    ce = torch.nn.CrossEntropyLoss()

    def emb(texts):
        enc = tok(texts, padding=True, truncation=True, max_length=MAXLEN, return_tensors="pt").to(DEV)
        h = model(**enc).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1).to(h.dtype)
        return torch.nn.functional.normalize((h * m).sum(1) / m.sum(1), dim=1)

    model.train()
    t0 = time.time()
    for s in range(steps):
        idx = perm[s * BS:(s + 1) * BS]
        with torch.autocast(DEV, dtype=torch.float16, enabled=DEV == "cuda"):
            ea, eb = emb([a_txt[i] for i in idx]), emb([b_txt[i] for i in idx])
            sim = (ea.float() @ eb.float().T) * SCALE
        lab = torch.arange(len(idx), device=DEV)
        loss = (ce(sim, lab) + ce(sim.T, lab)) / 2   # duplicates of the same S1 in a batch are rare at this sample size
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
        if s % 200 == 0:
            el = time.time() - t0
            log(f"step {s}/{steps} loss {loss.item():.4f} eta {el / (s + 1) * (steps - s) / 60:.0f} min")
    model.eval()


def main():
    rec = pl.read_parquet(f"{DATA}/train_records.parquet", columns=["id", "src", "name", "addr", "country"])
    rec = rec.select("id", "src", pl.col("country").cast(pl.Utf8),
                     ("query: " + pl.col("name") + " | " + pl.col("addr")).str.slice(0, 250).alias("t"))
    gt = pl.read_parquet(f"{DATA}/train_gt.parquet")
    s1_eval = gt.filter(pl.col("s1") % 8 == 7).select("s1").unique()
    s1_eval = s1_eval.sample(min(N_EVAL, s1_eval.height), seed=0)
    truth = gt.join(s1_eval, on="s1", how="semi")
    tr = gt.filter(pl.col("s1") % 8 != 7).sample(min(N_TRAIN, gt.height), seed=1)
    q = rec.join(s1_eval.rename({"s1": "id"}), on="id", how="semi")
    others = rec.filter(pl.col("src") != 1).join(truth.select("id"), on="id", how="anti")
    db = pl.concat([rec.join(truth.select("id"), on="id", how="semi"),
                    others.sample(min(N_DB, others.height), seed=2)]).select("id", "country", "t")
    log(f"eval S1 {q.height:,} | eval true pairs {truth.height:,} | db {db.height:,} | train pairs {tr.height:,}")

    tok = AutoTokenizer.from_pretrained(E5)
    model = AutoModel.from_pretrained(E5).to(DEV)
    base = recall_at(model, tok, q, db, truth)
    log("BASELINE e5 recall@5/10/20:", base)

    pairs = (tr.join(rec.select(pl.col("id").alias("s1"), pl.col("t").alias("ta")), on="s1")
             .join(rec.select("id", pl.col("t").alias("tb")), on="id"))   # order irrelevant: shuffled below
    a_txt, b_txt = pairs["ta"].to_list(), pairs["tb"].to_list()
    del rec, pairs
    finetune(model, tok, a_txt, b_txt)
    model.save_pretrained(OUT_MODEL)
    tok.save_pretrained(OUT_MODEL)
    ft = recall_at(model, tok, q, db, truth)
    log("FINE-TUNED e5 recall@5/10/20:", ft)
    log(f"DELTA recall@20: {ft[20] - base[20]:+.4f}   (go if the fine-tuned model clearly beats the baseline)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/src/s18_ftcos.py
"""Run E: fine-tuned e5 cosine as a pair FEATURE, leak-free (Fable review 2). Run C only used the fine-tuned model to add
candidates, because a model fine-tuned on train pairs scores those pairs better than unseen (test) pairs.
Fix: two models, split like the two XGBoost folds of s4_train (FOLDS = S1 buckets [0,2,4,6] / [1,3,5,7]):
  m{k} = original e5 fine-tuned (s17 recipe) on N_TRAIN (S1, match) pairs whose S1 bucket is in FOLDS[k].
  train pair in bucket j of fold f -> cosine from m{1-f}, a model that never saw bucket j.
  test pair -> both, as columns <feat>_m0 / <feat>_m1 (+ <feat> = m0). s4_train's stacked test path gives booster g
  (trained on FOLDS[g], whose rows carry m{1-g} cosines) the columns <feat>_m{1-g}: the same model it was trained with.
Features per union pair (s10_v3.union_pairs = keys + e5 kNN + fine-tuned kNN):
  ftc (cosine), ftc_rk / ftc_gap (rank / gap to the best within the S1), ftc_rk_id / ftc_gap_id (within the S2/S3 record's
  candidates; on train each pair keeps its own leak-free cosine, so a record's rivals may come from the other model).
Steps (each skips finished outputs):
  train <k>      : fine-tune m{k} -> WORK/e5ft_m{k}; recall check on held-out S1 of the OTHER fold (must beat the original e5)
  encode <split> : every record with m0 and m1, tokenized once (m0 on cuda:0, m1 on cuda:1 if present)
                   -> WORK/{split}_embft{k}.f16 (rows in length order) + {split}_embft_pos.npy (records row -> memmap row)
  feats <split>  : -> WORK/ftc_{split}_{j}.parquet
usage: s18_ftcos.py train <k> | encode <split> | feats <split>
env: BER_E5 (original e5 dir), BER_N_TRAIN (300k), BER_N_EVAL (20k), BER_N_DB (300k)"""
import os
import sys
import time

import numpy as np
import polars as pl
import torch
from transformers import AutoModel, AutoTokenizer

from common import WORK, done, timer
from s2_block import BUCKETS
from s4_train import FOLDS
from s10_v3 import union_pairs
import s17_ft_retrieval as s17

E5 = os.environ.get("BER_E5", "/kaggle/input/multilingual-e5-small")
N_TRAIN = int(os.environ.get("BER_N_TRAIN", 300_000))
N_EVAL = int(os.environ.get("BER_N_EVAL", 20_000))
N_DB = int(os.environ.get("BER_N_DB", 300_000))
DIM, MAXLEN, BS = 384, 64, 1024
FEATS = ["ftc", "ftc_rk", "ftc_gap", "ftc_rk_id", "ftc_gap_id"]


def model_dir(k):
    return WORK / f"e5ft_m{k}"


def emb_path(split, k):
    return WORK / f"{split}_embft{k}.f16"


def pos_path(split):
    return WORK / f"{split}_embft_pos.npy"


def ftc_path(split, j):
    return WORK / f"ftc_{split}_{j}.parquet"


def fold_of(j):
    return 0 if j in FOLDS[0] else 1


TXT = ("query: " + pl.col("name") + " | " + pl.col("addr")).str.slice(0, 250)   # same text as e1_emb / s17
MIN_GAIN = float(os.environ.get("BER_FT_MIN_GAIN", 0.0))   # recall@20 gain a fold model must show (local smoke: -1)


def train(k):
    if done(model_dir(k) / "model.safetensors"):
        return
    with timer(f"fine-tune m{k} on S1 buckets {FOLDS[k]}"):
        rec = pl.read_parquet(WORK / "train_records.parquet", columns=["id", "src", "name", "addr", "country"])
        rec = rec.select("id", "src", pl.col("country").cast(pl.Utf8), TXT.alias("t"))
        gt = pl.read_parquet(WORK / "train_gt.parquet")
        mine = (pl.col("s1") % BUCKETS).is_in(FOLDS[k])
        tr = gt.filter(mine).sample(min(N_TRAIN, gt.filter(mine).height), seed=11 + k)
        s1_eval = gt.filter(~mine).select("s1").unique()
        s1_eval = s1_eval.sample(min(N_EVAL, s1_eval.height), seed=21 + k)
        truth = gt.join(s1_eval, on="s1", how="semi")
        q = rec.join(s1_eval.rename({"s1": "id"}), on="id", how="semi")
        others = rec.filter(pl.col("src") != 1).join(truth.select("id"), on="id", how="anti")
        db = pl.concat([rec.join(truth.select("id"), on="id", how="semi"),
                        others.sample(min(N_DB, others.height), seed=31 + k)]).select("id", "country", "t")
        s17.log(f"m{k}: train pairs {tr.height:,} | eval S1 {q.height:,} (other fold) | eval pairs {truth.height:,} | db {db.height:,}")
        pairs = (tr.join(rec.select(pl.col("id").alias("s1"), pl.col("t").alias("ta")), on="s1")
                 .join(rec.select("id", pl.col("t").alias("tb")), on="id"))
        a_txt, b_txt = pairs["ta"].to_list(), pairs["tb"].to_list()
        del rec, pairs
        tok = AutoTokenizer.from_pretrained(E5)
        model = AutoModel.from_pretrained(E5).to(s17.DEV)
        base = s17.recall_at(model, tok, q, db, truth)
        s17.log(f"m{k} BASELINE original e5 recall@5/10/20: {base}")
        s17.finetune(model, tok, a_txt, b_txt, seed=41 + k)
        ft = s17.recall_at(model, tok, q, db, truth)
        s17.log(f"m{k} FINE-TUNED recall@5/10/20: {ft}  (delta @20 {ft[20] - base[20]:+.4f})")
        assert ft[20] - base[20] > MIN_GAIN, f"fine-tuned m{k} is not better than the original e5 -> refusing to use it"
        tmp = WORK / f"e5ft_m{k}.part"
        model.save_pretrained(tmp)
        tok.save_pretrained(tmp)
        os.replace(tmp, model_dir(k))


@torch.no_grad()
def encode(split):
    if all(done(emb_path(split, k)) for k in (0, 1)) and done(pos_path(split)):
        return
    with timer(f"encode {split} with m0 + m1"):
        r = pl.read_parquet(WORK / f"{split}_records.parquet", columns=["id", "name", "addr"])
        txt = r.select(TXT).to_series().to_list()
        del r
        n = len(txt)
        order = np.argsort(np.fromiter((len(t) for t in txt), dtype=np.int32, count=n), kind="stable")
        ng = torch.cuda.device_count()
        devs = [f"cuda:{min(k, ng - 1)}" if ng else "cpu" for k in (0, 1)]
        tok = AutoTokenizer.from_pretrained(model_dir(0))
        models = [AutoModel.from_pretrained(model_dir(k)).to(devs[k]).half().eval() for k in (0, 1)]
        outs = [np.memmap(emb_path(split, k).with_suffix(".part"), dtype=np.float16, mode="w+", shape=(n, DIM)) for k in (0, 1)]
        t0 = time.time()
        for s in range(0, n, BS):
            idx = order[s:s + BS]
            enc = tok([txt[i] for i in idx], padding=True, truncation=True, max_length=MAXLEN, return_tensors="pt")
            es = []
            for k in (0, 1):
                e = {a: b.to(devs[k], non_blocking=True) for a, b in enc.items()}
                h = models[k](**e).last_hidden_state
                m = e["attention_mask"].unsqueeze(-1).to(h.dtype)
                es.append(torch.nn.functional.normalize(((h * m).sum(1) / m.sum(1)).float(), dim=1).half())
            for k in (0, 1):
                outs[k][s:s + len(idx)] = es[k].cpu().numpy()   # memmap row = position in length order
            if s % (BS * 500) == 0:
                el = time.time() - t0
                print(f"  {s:,}/{n:,} {(s + len(idx)) / max(el, 1e-6):.0f}/s eta {el / (s + len(idx)) * (n - s) / 60:.0f} min", flush=True)
        for k in (0, 1):
            outs[k].flush()
        del outs
        pos = np.empty(n, dtype=np.int64)
        pos[order] = np.arange(n, dtype=np.int64)   # records row -> memmap row
        np.save(pos_path(split).with_suffix(".part.npy"), pos)
        for k in (0, 1):
            os.replace(emb_path(split, k).with_suffix(".part"), emb_path(split, k))
        os.replace(pos_path(split).with_suffix(".part.npy"), pos_path(split))


def pair_cos(E, u, dev):
    """cosine(S1, candidate) for pairs u [p1, p2] (memmap rows); u must be sorted by p2 (near-sequential reads)."""
    s1_rows = np.unique(u["p1"].to_numpy())
    S = torch.from_numpy(np.ascontiguousarray(E[s1_rows])).to(dev).float()
    loc = np.searchsorted(s1_rows, u["p1"].to_numpy()).astype(np.int64)
    p2 = u["p2"].to_numpy().astype(np.int64)
    out = np.empty(u.height, dtype=np.float32)
    for i in range(0, u.height, 500_000):
        b = torch.from_numpy(np.ascontiguousarray(E[p2[i:i + 500_000]])).to(dev).float()
        out[i:i + 500_000] = (S[torch.from_numpy(loc[i:i + 500_000]).to(dev)] * b).sum(1).cpu().numpy()
    del S
    return out


def derived(c, sfx):
    """rank / gap within the S1 and within the record, for cosine column c; output names FEATS + sfx."""
    return [pl.col(c).alias("ftc" + sfx),
            pl.col(c).rank("min", descending=True).over("s1").cast(pl.Int16).alias("ftc_rk" + sfx),
            (pl.col(c).max().over("s1") - pl.col(c)).cast(pl.Float32).alias("ftc_gap" + sfx),
            pl.col(c).rank("min", descending=True).over("id").cast(pl.Int16).alias("ftc_rk_id" + sfx),
            (pl.col(c).max().over("id") - pl.col(c)).cast(pl.Float32).alias("ftc_gap_id" + sfx)]


def feats(split):
    if all(done(ftc_path(split, j)) for j in range(BUCKETS)):
        return
    with timer(f"ftc features {split}"):
        n = len(np.load(pos_path(split)))
        Es = [np.memmap(emb_path(split, k), dtype=np.float16, mode="r", shape=(n, DIM)) for k in (0, 1)]
        ids = pl.read_parquet(WORK / f"{split}_records.parquet", columns=["id"])["id"]
        row = pl.DataFrame({"id": ids, "p": np.load(pos_path(split))})
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        parts = []
        for j in range(BUCKETS):
            u = (union_pairs(split, j).join(row.rename({"id": "s1", "p": "p1"}), on="s1")
                 .join(row.rename({"p": "p2"}), on="id").sort("p2"))
            # train: one leak-free cosine per pair (model of the other fold); test: both models
            ks = [1 - fold_of(j)] if split == "train" else [0, 1]
            u = u.with_columns([pl.Series(f"c{k}", pair_cos(Es[k], u, dev)) for k in ks])
            u = u.select("s1", "id", *[f"c{k}" for k in ks])
            if split == "train":
                u = u.rename({f"c{ks[0]}": "c"})
            parts.append(u)
            print(f"  cosines bucket {j}: {u.height:,} pairs", flush=True)
        x = pl.concat(parts)
        del parts
        if split == "train":
            x = x.select("s1", "id", *derived("c", ""))
        else:
            x = x.select("s1", "id", *derived("c0", "_m0"), *derived("c1", "_m1"))
            x = x.with_columns([pl.col(f + "_m0").alias(f) for f in FEATS])   # base column (renamed per booster in s4_train)
        for j in range(BUCKETS):
            tmp = ftc_path(split, j).with_suffix(".tmp")
            x.filter(pl.col("s1") % BUCKETS == j).write_parquet(tmp)
            os.replace(tmp, ftc_path(split, j))
        del x


if __name__ == "__main__":
    step, arg = sys.argv[1], sys.argv[2]
    {"train": lambda a: train(int(a)), "encode": encode, "feats": feats}[step](arg)

In [ ]:
%%writefile /kaggle/working/src/s19_preflight.py
"""Preflight for run E (it may be launched by a teammate): opens every input the run reads (parquet footer, npy header),
checks embedding sizes, 2 GPUs, the original e5 (in the inputs, or Hugging Face reachable), disk and RAM.
Prints one line 'PREFLIGHT OK: ...' or 'PREFLIGHT FAILED' + the full list of problems (exit 1), in about a minute.
env: BER_WORK (linked inputs), E5 (original e5 dir found in the inputs, may be empty)"""
import os
import shutil
import urllib.request

import numpy as np
import polars as pl
import torch

W = os.environ["BER_WORK"]
need = ["train_gt.parquet", "test_source1.tsv", "validate_submission.py"]
for s in ("train", "test"):
    need += [f"{s}_{x}.parquet" for x in ("records", "norm2", "norm3", "idf_ah", "idf_ch")] + [f"{s}_emb.f16", f"{s}_emb_ids.npy"]
    need += [f"{s}_{b}_{j}.parquet" for b in ("cand2", "feat2") for j in range(8)]
    need += [f"{b}_{s}_{j}.parquet" for b in ("knn", "knnft", "ce_v3") for j in range(8)]
SRC = {"knnft": "Output of KAGGLE_runC (Add Input -> Your Work -> Notebooks)", "knn_": "dataset mlc26-emb",
       "_emb": "dataset mlc26-emb", "ce_v3": "dataset mlc26-v3"}

bad = []
for f in need:
    p = os.path.join(W, f)
    try:
        if f.endswith(".parquet"):
            pl.read_parquet_schema(p)
        elif f.endswith(".npy"):
            np.load(p, mmap_mode="r")
        else:
            with open(p, "rb") as fh:
                fh.read(16)
    except Exception as e:
        bad.append(f"  {f}: {type(e).__name__}  (comes from {next((v for k, v in SRC.items() if k in f), 'dataset mlc26-core')})")
for s in ("train", "test"):
    try:
        n, sz = len(np.load(f"{W}/{s}_emb_ids.npy")), os.path.getsize(f"{W}/{s}_emb.f16")
        if sz != n * 384 * 2:
            bad.append(f"  {s}_emb.f16 is {sz:,} bytes, expected {n * 384 * 2:,} (incomplete mlc26-emb?)")
    except Exception:
        pass   # already listed above
ng = torch.cuda.device_count()
MIN_GPUS = int(os.environ.get("BER_MIN_GPUS", 2))   # local rehearsal on a 1-GPU laptop: 1
if ng < MIN_GPUS:
    bad.append(f"  GPUs visible: {ng}, need {MIN_GPUS} (Session options -> Accelerator -> GPU T4 x2)")
e5 = os.environ.get("E5", "")
if e5 and os.path.exists(os.path.join(e5, "model.safetensors")):
    e5msg = f"original e5 in the inputs ({e5})"
else:
    try:
        urllib.request.urlopen("https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/config.json", timeout=15)
        e5msg = "original e5 not in the inputs, Hugging Face reachable -> will download it"
    except Exception as e:
        bad.append(f"  original e5: not in the inputs and Hugging Face unreachable ({type(e).__name__}) -> Session options -> Internet On")
        e5msg = "MISSING"
free = shutil.disk_usage(W).free / 2**30
try:
    ram = f"{os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30:.0f} GB"
except (AttributeError, ValueError):   # not Linux (local rehearsal)
    ram = "n/a"
if bad:
    print(f"PREFLIGHT FAILED: {len(bad)} problem(s). Fix them and run again:", *bad, sep="\n", flush=True)
    raise SystemExit(1)
print(f"PREFLIGHT OK: all {len(need)} input files found and readable | GPUs {ng} ({torch.cuda.get_device_name(0)}) | {e5msg}"
      f" | disk free {free:.0f} GB | RAM {ram}", flush=True)

In [ ]:
%%writefile /kaggle/working/src/s20_rules.py
"""Run R: decode-time rules for generic-name false positives (Fable review 3), validated on labelled train OOF before use.
France looks precision-limited on generic names (shared by several S1); US/India S1 with a shared core name are France in
miniature WITH labels. A rule may be used only if, on train OOF at the submitted setting (t 0.80), it raises macro F0.5 and the
pairs it drops are mostly false (precision < DROP_PREC_MAX; dropping a set helps F0.5 when its precision < ~0.78).
  oof           : OOF p7 on train from run D's saved stage-7 models (xgb_s7_v3_f*.json) -> must reproduce D's OOF (0.98702)
  rules         : every rule variant on OOF -> table + WORK/rules.json (best passing variant per rule family, then combined)
  apply <name> <rules.json> <stage...> : test p of the stages (mean), t 0.80 / France 0.90, passing rules -> OUT/<name>/
Rule families (generic = the S1's core name is shared by >= 2 S1 of its country, norm3 core_n1 >= 2):
  num  : generic, both have house numbers, none in common, p < k      -> drop
  noad : generic, the S2/S3 record has no address, p < k              -> drop
  cap  : more matches for one S1 than the train GT ever has (q-quantile) -> keep the top by p
  amb  : the record's top-2 S1 (all candidates) share the same generic core name and p1 - p2 < d -> the record gets no match"""
import json
import sys

import numpy as np
import polars as pl

from common import OUT, WORK, done, f05, timer
from s2_block import BUCKETS
import s4_train as s4
from s15_blend import blended, LOW
from s6_write import to_tsv

T, T_FR = 0.80, 0.90
DROP_PREC_MAX = 0.70
LOWP = 0.5   # pairs below this can never be assigned at t 0.80 and are too far below p1 to matter for 'amb'
GRID = {"num": [0.97, 0.99, 1.01], "noad": [0.97, 0.99, 1.01], "cap": [0.99, 0.999], "amb": [0.05, 0.10]}


def oof():
    import xgboost as xgb
    bsts = [xgb.Booster(model_file=str(s4.model_path(7, f))) for f in range(2)]
    for b in bsts:
        b.set_param({"device": "cuda"})
    cols = bsts[0].feature_names
    have = s4.feature_cols(7)
    assert cols == bsts[1].feature_names and sorted(cols) == sorted(have), f"feature mismatch: {set(cols) ^ set(have)}"
    print(f"stage 7 models from run D: {len(cols)} features, all present", flush=True)
    for f, va in enumerate(s4.FOLDS):
        for j in va:
            if not done(s4.p_path(7, "train", j)):
                with timer(f"oof stage 7 bucket {j}"):
                    s4.predict([bsts[1 - f]], "train", j, 7, cols).write_parquet(s4.p_path(7, "train", j))
    s4.evaluate(7)
    (pl.concat([pl.read_parquet(s4.p_path(7, "train", j), columns=["s1", "id", "p"]).filter(pl.col("p") >= LOWP) for j in range(BUCKETS)])
     .write_parquet(OUT / "state" / "oof_p7_ge05.parquet"))


def oof_pred():
    """OOF p7 >= LOWP: from WORK (this run) or from a previous run R's Output (state/oof_p7_ge05.parquet linked into WORK)."""
    if all(done(s4.p_path(7, "train", j)) for j in range(BUCKETS)):
        return pl.concat([pl.read_parquet(s4.p_path(7, "train", j), columns=["s1", "id", "p"]).filter(pl.col("p") >= LOWP)
                          for j in range(BUCKETS)])
    return pl.read_parquet(WORK / "oof_p7_ge05.parquet")


def context(split):
    n = pl.read_parquet(WORK / f"{split}_norm3.parquet", columns=["id", "src", "country", "core", "core_n1", "nums", "addr_n"])
    s1 = n.filter(pl.col("src") == 1).select(pl.col("id").alias("s1"), pl.col("country").cast(pl.Utf8), pl.col("core").alias("s1_core"),
                                             pl.col("core_n1").alias("gen"), pl.col("nums").alias("s1_nums"))
    rec = n.filter(pl.col("src") != 1).select("id", pl.col("nums").alias("r_nums"),
                                              (pl.col("addr_n").fill_null("").str.strip_chars() == "").alias("r_noaddr"))
    return s1, rec


def decode(pred, t_fr=None):
    """Same as s4_train.assign / s15_blend _fr90: keep p >= t (France: t_fr), each record -> its best S1. Keeps p."""
    q = pred.filter(pl.when(pl.col("country") == "France").then(pl.col("p") >= (t_fr or T)).otherwise(pl.col("p") >= T))
    return q.sort("p", descending=True).unique("id", keep="first")


def ambiguous(pred, s1ctx):
    """Records whose two best S1 (by p, all candidates >= LOWP) share the same generic core name: [id, gap]."""
    top = (pred.sort("p", descending=True).group_by("id", maintain_order=True).head(2)
           .join(s1ctx.select("s1", "s1_core", "gen"), on="s1")
           .sort(["id", "p"], descending=[False, True])
           .group_by("id").agg(pl.len().alias("k"), pl.col("p").first().alias("p1"), pl.col("p").last().alias("p2"),
                               pl.col("s1_core").first().alias("c1"), pl.col("s1_core").last().alias("c2"),
                               pl.col("gen").first().alias("g1")))
    return (top.filter((pl.col("k") == 2) & (pl.col("c1") == pl.col("c2")) & (pl.col("g1") >= 2))
            .select("id", (pl.col("p1") - pl.col("p2")).alias("amb_gap")))


def drop_mask(fam, v, a, amb, cap_n):
    """Boolean expression over assigned pairs a (with context) = pairs this rule drops."""
    gen = pl.col("gen") >= 2
    if fam == "num":
        both = (pl.col("s1_nums").list.len() > 0) & (pl.col("r_nums").list.len() > 0)
        return gen & both & (pl.col("s1_nums").list.set_intersection(pl.col("r_nums")).list.len() == 0) & (pl.col("p") < v)
    if fam == "noad":
        return gen & pl.col("r_noaddr") & (pl.col("p") < v)
    if fam == "cap":
        return pl.col("p").rank("ordinal", descending=True).over("s1") > cap_n[v]
    if fam == "amb":
        return pl.col("amb_gap") < v   # column joined by with_ctx (null = record not ambiguous)
    raise ValueError(fam)


def with_ctx(a, s1ctx, rec, amb):
    return a.join(s1ctx.drop("country"), on="s1", how="left").join(rec, on="id", how="left").join(amb, on="id", how="left")


def apply_rules(a, sel, amb, cap_n):
    m = pl.lit(False)
    for fam, v in sel.items():
        m = m | drop_mask(fam, v, a, amb, cap_n).fill_null(False)   # null (missing context) must never drop a match
    return a.with_columns(m.alias("_drop"))


def rules():
    with timer("rules on train OOF"):
        s1ctx, rec = context("train")
        pred = oof_pred().join(s1ctx.select("s1", "country"), on="s1")
        truth = pl.read_parquet(WORK / "train_gt.parquet")
        s1_all = s1ctx.select("s1", "country", "gen")
        n_true = truth.group_by("s1").len()["len"]
        cap_n = {q: int(np.ceil(n_true.quantile(q))) for q in GRID["cap"]}
        amb = ambiguous(pred, s1ctx)
        a = with_ctx(decode(pred), s1ctx, rec, amb)
        (a.filter(pl.col("p") < 0.99).join(truth.with_columns(pl.lit(1, pl.Int8).alias("y")), on=["s1", "id"], how="left")
         .select("s1", "id", "p", "country", "gen", pl.col("y").fill_null(0))
         .write_parquet(OUT / "state" / "oof_band.parquet"))   # labelled uncertain matches (for an LLM-veto check)
        subsets = {"all": s1_all, "US": s1_all.filter(pl.col("country") == "US"), "India": s1_all.filter(pl.col("country") == "India"),
                   "generic": s1_all.filter(pl.col("gen") >= 2)}

        def score(pairs):
            p2 = pairs.select("s1", "id")
            return {k: f05(v, p2.join(v.select("s1"), on="s1", how="semi"), truth.join(v.select("s1"), on="s1", how="semi"))
                    for k, v in subsets.items()}

        base = score(a)
        print(f"baseline OOF at t {T}: " + " | ".join(f"{k} {v:.5f}" for k, v in base.items())
              + f" | generic S1 {subsets['generic'].height:,} of {s1_all.height:,} | caps {cap_n}", flush=True)
        rows, best = [], {}
        for fam, vals in GRID.items():
            for v in vals:
                x = apply_rules(a, {fam: v}, amb, cap_n)
                d = x.filter(pl.col("_drop")).select("s1", "id")
                prec = d.join(truth, on=["s1", "id"], how="semi").height / max(d.height, 1)
                sc = score(x.filter(~pl.col("_drop")))
                dF = {k: sc[k] - base[k] for k in sc}
                ok = dF["all"] > 0 and prec < DROP_PREC_MAX and d.height > 0
                rows.append((fam, v, d.height, prec, dF, ok))
                print(f"  {fam:5s} {v:<6} dropped {d.height:>8,}  precision {prec:.3f}  dF all {dF['all']:+.5f}  US {dF['US']:+.5f}"
                      f"  India {dF['India']:+.5f}  generic {dF['generic']:+.5f}  {'PASS' if ok else '-'}", flush=True)
                if ok and dF["all"] > best.get(fam, (None, 0))[1]:
                    best[fam] = (v, dF["all"])
        sel = {fam: v for fam, (v, _) in best.items()}
        comb = score(apply_rules(a, sel, amb, cap_n).filter(~pl.col("_drop"))) if sel else base
        print(f"SELECTED {sel} -> combined OOF at t {T}: " + " | ".join(f"{k} {comb[k]:.5f} ({comb[k] - base[k]:+.5f})" for k in comb),
              flush=True)
        json.dump({"selected": sel, "cap_n": {str(k): v for k, v in cap_n.items()}, "baseline": base, "combined": comb,
                   "table": [(f, v, n, p, d, ok) for f, v, n, p, d, ok in rows]}, open(WORK / "rules.json", "w"), indent=1)


def apply(name, rules_json, stages):
    r = json.load(open(rules_json))
    sel = r["selected"]
    cap_n = {float(k): v for k, v in r["cap_n"].items()}
    with timer(f"apply rules {sel} to stages {stages} -> {name}"):
        s1ctx, rec = context("test")
        tp = (pl.concat([blended(stages, "test", j, LOW) for j in range(BUCKETS)]).filter(pl.col("p") >= LOWP)
              .join(s1ctx.select("s1", "country"), on="s1"))
        a = with_ctx(decode(tp, T_FR), s1ctx, rec, ambiguous(tp, s1ctx))
        x = apply_rules(a, sel, None, cap_n)
        n_s1 = s1ctx.group_by("country").len("n_s1")
        print((x.group_by("country").agg(pl.len().alias("matched"), pl.col("_drop").sum().alias("dropped"))
               .join(n_s1, on="country").with_columns((pl.col("matched") / pl.col("n_s1")).round(3).alias("matches_per_S1"),
                                                      ((pl.col("matched") - pl.col("dropped")) / pl.col("n_s1")).round(3).alias("after_rules"))
               .sort("country")), flush=True)
        m = x.filter(~pl.col("_drop")).select("s1", "id")
        s1_all = (pl.scan_parquet(WORK / "test_norm2.parquet").filter(pl.col("src") == 1)
                  .select(pl.col("id").alias("s1")).sort("s1").collect())
        d = OUT / name
        d.mkdir(parents=True, exist_ok=True)
        to_tsv(s1_all, m, "matched_entity_ids", d / "matching_results.tsv")
        print(f"  wrote {d}/matching_results.tsv  matched pairs {m.height:,} (before rules {a.height:,})", flush=True)


def selftest():
    s1ctx = pl.DataFrame({"s1": [1, 2, 3], "country": ["US", "US", "France"], "s1_core": ["acme", "acme", "zeta"],
                          "gen": [2, 2, 1], "s1_nums": [["12"], ["40"], []]})
    rec = pl.DataFrame({"id": [10, 11, 12, 13], "r_nums": [["12"], ["99"], [], ["5"]], "r_noaddr": [False, False, True, False]})
    pred = pl.DataFrame({"s1": [1, 2, 1, 1, 3, 2], "id": [10, 10, 11, 12, 13, 13], "p": [0.95, 0.93, 0.85, 0.9, 0.97, 0.6],
                         "country": ["US", "US", "US", "US", "France", "US"]})
    amb = ambiguous(pred, s1ctx)
    assert amb.sort("id")["id"].to_list() == [10] and abs(amb["amb_gap"][0] - 0.02) < 1e-9, amb
    a = with_ctx(decode(pred, T_FR), s1ctx, rec, amb)
    assert sorted(a["id"].to_list()) == [10, 11, 12, 13], a
    cap_n = {0.99: 1}
    got = {fam: sorted(apply_rules(a, {fam: v}, None, cap_n).filter(pl.col("_drop"))["id"].to_list())
           for fam, v in (("num", 1.01), ("noad", 1.01), ("cap", 0.99), ("amb", 0.05))}
    assert got == {"num": [11], "noad": [12], "cap": [11, 12], "amb": [10]}, got
    x = apply_rules(a, {"num": 1.01, "amb": 0.05}, None, cap_n)
    assert x["_drop"].null_count() == 0 and sorted(x.filter(~pl.col("_drop"))["id"].to_list()) == [12, 13]
    truth = pl.DataFrame({"s1": [1, 3], "id": [10, 13]})
    f = f05(s1ctx.select("s1"), x.filter(~pl.col("_drop")).select("s1", "id"), truth)
    print(f"SELFTEST OK (polars {pl.__version__}): ambiguous / decode / 4 rule families / f05 = {f:.4f}", flush=True)


if __name__ == "__main__":
    if sys.argv[1] == "selftest":
        selftest()
    elif sys.argv[1] == "oof":
        oof()
    elif sys.argv[1] == "rules":
        rules()
    elif sys.argv[1] == "apply":
        apply(sys.argv[2], sys.argv[3], [int(s) for s in sys.argv[4:]])

In [ ]:
%%writefile /kaggle/working/src/s21_efdecode.py
"""Per-S1 expected-F0.5 decoding on saved predictions (Fable review 4), tuned on labelled OOF before any use.
After the global assignment (each S2/S3 record -> its best S1, p >= floor), each S1 keeps the top-k of its candidates that
maximise E[F0.5 | k] ~= 1.25 * sum_{i<=k} q_i / (0.25 * sum_i q_i + k), or nothing when P(no true match) = prod(1 - q_i) is higher
(singletons score 1 only when empty). q = p ** gamma (calibration). France may use its own (floor, gamma): its probabilities
proved over-confident on the LB (France 0.80 -> 0.90 threshold: France F +0.0013). Same maths as s9_decode.decode.
  selftest
  tune <oof.parquet>                          : grid of (floor, gamma) on OOF vs the global t 0.80 -> prints best + delta
  apply <name> <floor> <gamma> <floor_fr> <gamma_fr> <stage...> : test p (mean of stages) -> OUT/<name>/matching_results.tsv"""
import sys

import numpy as np
import polars as pl

from common import OUT, WORK, f05, timer
from s2_block import BUCKETS

T = 0.80
FLOORS = [0.5, 0.6, 0.7, 0.8]
GAMMAS = [1.0, 1.25, 1.5, 2.0]


def assigned(pred, floor, floor_fr=None):
    fr = pl.col("country") == "France" if "country" in pred.columns else pl.lit(False)
    q = pred.filter(pl.when(fr).then(pl.col("p") >= (floor_fr if floor_fr is not None else floor)).otherwise(pl.col("p") >= floor))
    return q.sort("p", descending=True).unique("id", keep="first")


def decode(pa, g, g_fr=None):
    fr = pl.col("country") == "France" if "country" in pa.columns else pl.lit(False)
    gam = pl.when(fr).then(pl.lit(g_fr if g_fr is not None else g)).otherwise(pl.lit(g))
    x = (pa.with_columns((pl.col("p").cast(pl.Float64).clip(1e-6, 1.0).log() * gam).exp().clip(1e-6, 1 - 1e-6).alias("q"))
         .sort(["s1", "q"], descending=[False, True])
         .with_columns(pl.col("q").cum_sum().over("s1").alias("cq"), pl.int_range(1, pl.len() + 1).over("s1").alias("k"),
                       pl.col("q").sum().over("s1").alias("sq"), (1 - pl.col("q")).log().sum().over("s1").exp().alias("p_empty")))
    x = x.with_columns((1.25 * pl.col("cq") / (0.25 * pl.col("sq") + pl.col("k"))).alias("ef"))
    best = (x.sort(["s1", "ef", "k"], descending=[False, True, False]).group_by("s1", maintain_order=True)
            .agg(pl.col("ef").first().alias("ef_best"), pl.col("k").first().alias("k_best"), pl.col("p_empty").first()))
    keep = best.filter(pl.col("ef_best") > pl.col("p_empty")).select("s1", "k_best")
    return x.join(keep, on="s1").filter(pl.col("k") <= pl.col("k_best")).select("s1", "id")


def global_t(pred, t, t_fr=None):
    return assigned(pred, t, t_fr).select("s1", "id")


def selftest():
    # S1 1: one sure match + one 0.6 candidate; S1 2: a lone 0.55 candidate (empty is worth 0.45); S1 3: nothing
    pred = pl.DataFrame({"s1": [1, 1, 2, 1], "id": [10, 11, 12, 13], "p": [0.99, 0.6, 0.55, 0.52], "country": ["US"] * 4})
    k = decode(assigned(pred, 0.5), 1.0).sort("id")
    assert k["id"].to_list() == [10, 12], k      # 0.6 on a sure cluster is below break-even (~0.7); lone 0.55 > 0.5 is kept
    assert global_t(pred, 0.8)["id"].to_list() == [10]
    fr = pred.with_columns(pl.lit("France").alias("country"))
    assert decode(assigned(fr, 0.5, 0.9), 1.0, 2.0)["id"].to_list() == [10]
    truth = pl.DataFrame({"s1": [1, 2], "id": [10, 12]})
    f = f05(pl.DataFrame({"s1": [1, 2, 3]}), k, truth)
    assert abs(f - 1.0) < 1e-9, f
    print(f"SELFTEST OK (polars {pl.__version__}): assign / decode / France params / f05", flush=True)


def tune(path):
    import polars as pl2   # noqa: F401  (same module; kept for clarity when run locally)
    oof = pl.read_parquet(path).select("s1", "id", "p")
    truth = pl.read_parquet(WORK / "train_gt.parquet")
    s1 = (pl.scan_parquet(WORK / "train_norm2.parquet").filter(pl.col("src") == 1)
          .select(pl.col("id").alias("s1"), pl.col("country").cast(pl.Utf8)).collect())
    oof = oof.join(s1, on="s1")
    base = f05(s1, global_t(oof, T), truth)
    print(f"global t {T}: {base:.5f}", flush=True)
    res = {}
    for fl in FLOORS:
        pa = assigned(oof, fl)
        for g in GAMMAS:
            res[(fl, g)] = f05(s1, decode(pa, g), truth)
            print(f"  floor {fl} gamma {g}: {res[(fl, g)]:.5f} ({res[(fl, g)] - base:+.5f})", flush=True)
    (bf, bg), bF = max(res.items(), key=lambda kv: kv[1])
    by = {c: (f05(s1.filter(pl.col("country") == c), decode(assigned(oof, bf), bg).join(s1.filter(pl.col("country") == c), on="s1", how="semi"),
                  truth.join(s1.filter(pl.col("country") == c), on="s1", how="semi")),
              f05(s1.filter(pl.col("country") == c), global_t(oof, T).join(s1.filter(pl.col("country") == c), on="s1", how="semi"),
                  truth.join(s1.filter(pl.col("country") == c), on="s1", how="semi"))) for c in ("US", "India")}
    print(f"BEST floor {bf} gamma {bg}: {bF:.5f} vs global t {T} {base:.5f} -> delta {bF - base:+.5f} | by country (ef, global) {by}",
          flush=True)


def apply(name, floor, g, floor_fr, g_fr, stages):
    from s15_blend import blended, LOW
    from s6_write import to_tsv
    with timer(f"expected-F decode {name}: floor {floor} gamma {g} | France floor {floor_fr} gamma {g_fr} | stages {stages}"):
        cty = pl.read_parquet(WORK / "test_records.parquet", columns=["id", "country"]).select(pl.col("id").alias("s1"), pl.col("country").cast(pl.Utf8))
        tp = pl.concat([blended(stages, "test", j, LOW) for j in range(BUCKETS)]).join(cty, on="s1")
        m = decode(assigned(tp, floor, floor_fr), g, g_fr)
        ref = global_t(tp, T, 0.90)
        n1 = cty.group_by("country").len("n_s1")
        stat = (m.join(cty, on="s1").group_by("country").len("ef").join(ref.join(cty, on="s1").group_by("country").len("fr90"), on="country")
                .join(n1, on="country").with_columns((pl.col("ef") / pl.col("n_s1")).round(3).alias("ef_per_S1"),
                                                     (pl.col("fr90") / pl.col("n_s1")).round(3).alias("fr90_per_S1")).sort("country"))
        print(stat, flush=True)
        s1_all = (pl.scan_parquet(WORK / "test_norm2.parquet").filter(pl.col("src") == 1).select(pl.col("id").alias("s1")).sort("s1").collect())
        d = OUT / name
        d.mkdir(parents=True, exist_ok=True)
        to_tsv(s1_all, m, "matched_entity_ids", d / "matching_results.tsv")
        print(f"  wrote {d}/matching_results.tsv  matched pairs {m.height:,} (global t 0.80 / France 0.90: {ref.height:,})", flush=True)


if __name__ == "__main__":
    a = sys.argv
    if a[1] == "selftest":
        selftest()
    elif a[1] == "tune":
        tune(a[2])
    elif a[1] == "apply":
        apply(a[2], float(a[3]), float(a[4]), float(a[5]), float(a[6]), [int(s) for s in a[7:]])

In [ ]:
%%writefile /kaggle/working/src/s22_stack.py
"""Set-level second stage on saved probabilities (last sprint). The pair model scores each (S1, record) alone; the error budget
shows the remaining loss is set-level (S1 missing one match, a record taken by the wrong same-name S1, another business at the same
address). Features use ONLY one model's p (>= 0.5) plus record hashes, so they are built identically on train OOF and on test:
  S1 side   : #candidates, #confident (p >= 0.9) overall / per source, rank of p, max / sum of p, #S1 sharing the name
  record    : #S1 candidates, rank, gap to its best S1, the best rival S1's p and #confident, record without address
  siblings  : the S1's other confident matches with the same exact address / name as this record; exact name / address = S1's
Train: run R's OOF p7 (state/oof_p7_ge05.parquet, single-model OOF). Test: run D's p7 per model (p_m0, p_m1) -> stacked separately
(each is a single-model p like the OOF) -> mean. Decode t 0.80 / France 0.90 (the LB-proven setting).
  selftest | run <name>"""
import sys

import numpy as np
import polars as pl

from common import OUT, WORK, f05, timer
from s2_block import BUCKETS

T, T_FR, CONF, LOWP = 0.80, 0.90, 0.9, 0.5
F = ["p", "s1_n", "s1_nconf", "s1_rk", "s1_pmax", "s1_psum", "r_n", "r_rk", "r_gap", "addr_eq_s1", "name_eq_s1", "s1_conf_s2",
     "s1_conf_s3", "sib_same_addr", "sib_same_name", "rival_nconf", "rival_p", "s1_name_n", "r_noaddr", "src"]
import os
USE_CE = os.environ.get("BER_STACK_CE") == "1"   # run X: + fold-wise cross-encoder logit on the uncertain band (s23, ceband_*)
if USE_CE:
    F = F + ["ce"]
PARAMS = dict(objective="binary:logistic", eta=0.1, max_depth=8, subsample=0.8, colsample_bytree=0.8, tree_method="hist",
              eval_metric="logloss", nthread=4, seed=0, device=os.environ.get("BER_STACK_DEVICE", "cpu"))
ROUNDS = 300
_NORM = lambda c: pl.col(c).fill_null("").str.to_lowercase().str.strip_chars()


def context(split):
    """S1 [s1, country, n1, a1, s1_name_n] and records [id, n2, a2, src, r_noaddr] from {split}_records.parquet."""
    r = pl.read_parquet(WORK / f"{split}_records.parquet", columns=["id", "src", "name", "addr", "country"])
    r = r.with_columns(_NORM("name").hash().alias("hn"), _NORM("addr").hash().alias("ha"), (_NORM("addr") == "").alias("noaddr"),
                       pl.col("country").cast(pl.Utf8))
    s1 = (r.filter(pl.col("src") == 1).select(pl.col("id").alias("s1"), "country", pl.col("hn").alias("n1"), pl.col("ha").alias("a1"))
          .with_columns(pl.len().over(["country", "n1"]).alias("s1_name_n")))
    rec = r.filter(pl.col("src") != 1).select("id", pl.col("hn").alias("n2"), pl.col("ha").alias("a2"), pl.col("src").cast(pl.Int8),
                                              pl.col("noaddr").alias("r_noaddr"))
    return s1, rec


def features(pr, s1, rec):
    """pr [s1, id, p]: ONE model's probabilities, p >= LOWP."""
    x = pr.join(s1, on="s1").join(rec, on="id")
    conf = pl.col("p") >= CONF
    x = x.with_columns(
        pl.len().over("s1").alias("s1_n"), conf.sum().over("s1").alias("s1_nconf"),
        pl.col("p").rank("ordinal", descending=True).over("s1").alias("s1_rk"),
        pl.col("p").max().over("s1").alias("s1_pmax"), pl.col("p").sum().over("s1").alias("s1_psum"),
        pl.len().over("id").alias("r_n"), pl.col("p").rank("ordinal", descending=True).over("id").alias("r_rk"),
        (pl.col("p").max().over("id") - pl.col("p")).alias("r_gap"),
        (pl.col("a2") == pl.col("a1")).cast(pl.Int8).alias("addr_eq_s1"), (pl.col("n2") == pl.col("n1")).cast(pl.Int8).alias("name_eq_s1"),
        (conf & (pl.col("src") == 2)).sum().over("s1").alias("s1_conf_s2"), (conf & (pl.col("src") == 3)).sum().over("s1").alias("s1_conf_s3"))
    sa = x.filter(conf).group_by("s1", "a2").len("sib_same_addr")
    sn = x.filter(conf).group_by("s1", "n2").len("sib_same_name")
    own = conf.cast(pl.Int64)
    x = (x.join(sa, on=["s1", "a2"], how="left").join(sn, on=["s1", "n2"], how="left")
         .with_columns((pl.col("sib_same_addr").fill_null(0).cast(pl.Int64) - own).alias("sib_same_addr"),
                       (pl.col("sib_same_name").fill_null(0).cast(pl.Int64) - own).alias("sib_same_name")))
    # the record's best rival S1 (the best S1 that is not this one): top-2 per record via first / last (no list.get)
    top = (x.select("id", "s1", "p", "s1_nconf").sort(["id", "p"], descending=[False, True]).group_by("id", maintain_order=True).head(2)
           .group_by("id").agg(pl.len().alias("k"), pl.col("s1").first().alias("t1"), pl.col("p").first().alias("tp1"),
                               pl.col("s1_nconf").first().alias("tn1"), pl.col("p").last().alias("tp2"), pl.col("s1_nconf").last().alias("tn2")))
    x = x.join(top, on="id").with_columns(
        pl.when(pl.col("t1") != pl.col("s1")).then(pl.col("tp1")).when(pl.col("k") == 2).then(pl.col("tp2")).otherwise(None).alias("rival_p"),
        pl.when(pl.col("t1") != pl.col("s1")).then(pl.col("tn1")).when(pl.col("k") == 2).then(pl.col("tn2")).otherwise(None).alias("rival_nconf"))
    return x.drop("k", "t1", "tp1", "tn1", "tp2", "tn2")


def ce_all(split):
    return pl.concat([pl.read_parquet(WORK / f"ceband_{split}_{j}.parquet", columns=["s1", "id", "ce"]) for j in range(BUCKETS)])


def matrix(x):
    return x.select(pl.col(F).cast(pl.Float32)).to_numpy()


def decode(x, col, t=T, t_fr=T_FR):
    q = x.filter(pl.when(pl.col("country") == "France").then(pl.col(col) >= t_fr).otherwise(pl.col(col) >= t))
    return q.sort(col, descending=True).unique("id", keep="first").select("s1", "id")


def selftest():
    s1 = pl.DataFrame({"s1": [1, 2], "country": ["US", "US"], "n1": [7, 7], "a1": [100, 200], "s1_name_n": [2, 2]})
    rec = pl.DataFrame({"id": [10, 11, 12], "n2": [7, 7, 7], "a2": [100, 100, 5], "src": [2, 3, 3], "r_noaddr": [False, False, True]}).with_columns(pl.col("src").cast(pl.Int8))
    pr = pl.DataFrame({"s1": [1, 1, 1, 2], "id": [10, 11, 12, 12], "p": [0.97, 0.95, 0.6, 0.7]})
    x = features(pr, s1, rec).sort("s1", "id")
    r = {(a, b): d for a, b, d in x.select("s1", "id", pl.struct(F).alias("d")).iter_rows()}
    assert r[(1, 10)]["sib_same_addr"] == 1 and r[(1, 11)]["sib_same_addr"] == 1 and r[(1, 12)]["sib_same_addr"] == 0, r
    assert r[(1, 12)]["rival_p"] == 0.7 and r[(2, 12)]["rival_p"] == 0.6 and r[(1, 10)]["rival_p"] is None, r
    assert r[(1, 12)]["r_n"] == 2 and r[(1, 10)]["s1_nconf"] == 2 and r[(2, 12)]["s1_nconf"] == 0 and r[(1, 12)]["r_noaddr"] == 1, r
    assert matrix(x).shape == (4, len(F)) and x["rival_nconf"].to_list() == [None, None, 0, 2], x
    print(f"SELFTEST OK (polars {pl.__version__}): stack features / rival / siblings", flush=True)


def run(name):
    import xgboost as xgb
    with timer("stack: train on run R's OOF"):
        s1, rec = context("train")
        truth = pl.read_parquet(WORK / "train_gt.parquet")
        oof = pl.read_parquet(WORK / "oof_p7_ge05.parquet").select("s1", "id", "p")
        x = features(oof, s1, rec).join(truth.with_columns(pl.lit(1, pl.Int8).alias("y")), on=["s1", "id"], how="left").with_columns(pl.col("y").fill_null(0))
        if USE_CE:
            x = x.join(ce_all("train"), on=["s1", "id"], how="left")
            print(f"  CE scored {x['ce'].is_not_null().sum():,} of {x.height:,} OOF pairs", flush=True)
        f0 = (pl.col("s1") % BUCKETS).is_in([0, 2, 4, 6])
        parts = []
        for f in (0, 1):
            tr, va = x.filter(f0 if f == 0 else ~f0), x.filter(~f0 if f == 0 else f0)
            b = xgb.train(PARAMS, xgb.DMatrix(matrix(tr), label=tr["y"].to_numpy(), feature_names=F), ROUNDS)
            parts.append(va.select("s1", "id", "p", "country").with_columns(pl.Series("p2", b.predict(xgb.DMatrix(matrix(va), feature_names=F)))))
        o = pl.concat(parts)
        s1_all = s1.select("s1", "country")
        base, st = f05(s1_all, decode(o, "p"), truth), f05(s1_all, decode(o, "p2"), truth)
        print(f"OOF (US+India) at t {T}: base p {base:.5f} | stacked p2 {st:.5f} | delta {st - base:+.5f}  (laptop run: +0.00078)", flush=True)
        for c in ("US", "India"):
            sc = s1_all.filter(pl.col("country") == c)
            tc = truth.join(sc, on="s1", how="semi")
            print(f"  {c}: base {f05(sc, decode(o, 'p').join(sc, on='s1', how='semi'), tc):.5f} | "
                  f"stacked {f05(sc, decode(o, 'p2').join(sc, on='s1', how='semi'), tc):.5f}", flush=True)
        import os
        assert st > base or os.environ.get("BER_STACK_FORCE") == "1", "stacking is not better on OOF -> not writing a submission"
        full = xgb.train(PARAMS, xgb.DMatrix(matrix(x), label=x["y"].to_numpy(), feature_names=F), ROUNDS)
        del x, parts, o
    with timer("stack: test (run D's p7, per model)"):
        s1t, rect = context("test")
        # pairs with both models' p < LOWP can never reach t (mean < LOWP < T): not read (same output, much less RAM)
        tp = pl.concat([pl.scan_parquet(WORK / f"p7_v3_test_{j}.parquet").select("s1", "id", "p_m0", "p_m1")
                        .filter((pl.col("p_m0") >= LOWP) | (pl.col("p_m1") >= LOWP)).collect() for j in range(BUCKETS)])
        outs = []
        for m in ("p_m0", "p_m1"):
            xm = features(tp.filter(pl.col(m) >= LOWP).select("s1", "id", pl.col(m).alias("p")), s1t, rect)
            if USE_CE:
                xm = (xm.join(ce_all("test"), on=["s1", "id"], how="left")
                      .with_columns(pl.when(pl.col("p") < 0.995).then(pl.col("ce")).otherwise(None).alias("ce")))
            outs.append(xm.select("s1", "id", pl.Series(f"s_{m}", full.predict(xgb.DMatrix(matrix(xm), feature_names=F)))))
        # a pair below LOWP for one model gets that model's raw p (it can't pass t anyway): mean over both models
        y = (tp.join(outs[0], on=["s1", "id"], how="left").join(outs[1], on=["s1", "id"], how="left")
             .with_columns(((pl.col("s_p_m0").fill_null(pl.col("p_m0")) + pl.col("s_p_m1").fill_null(pl.col("p_m1"))) / 2).alias("p2"),
                           ((pl.col("p_m0") + pl.col("p_m1")) / 2).alias("p"))
             .join(s1t.select("s1", "country"), on="s1"))
        m2, m1 = decode(y, "p2"), decode(y, "p")
        n1 = s1t.group_by("country").len("n_s1")
        print(m2.join(s1t.select("s1", "country"), on="s1").group_by("country").len("stack")
              .join(m1.join(s1t.select("s1", "country"), on="s1").group_by("country").len("p7_plain"), on="country").join(n1, on="country")
              .with_columns((pl.col("stack") / pl.col("n_s1")).round(3).alias("stack_per_S1"),
                            (pl.col("p7_plain") / pl.col("n_s1")).round(3).alias("plain_per_S1")).sort("country"), flush=True)
        from s6_write import to_tsv
        s1_all = (pl.scan_parquet(WORK / "test_norm2.parquet").filter(pl.col("src") == 1).select(pl.col("id").alias("s1")).sort("s1").collect())
        d = OUT / name
        d.mkdir(parents=True, exist_ok=True)
        to_tsv(s1_all, m2, "matched_entity_ids", d / "matching_results.tsv")
        print(f"  wrote {d}/matching_results.tsv  matched pairs {m2.height:,} (plain p7 at the same decode: {m1.height:,})", flush=True)


if __name__ == "__main__":
    selftest() if sys.argv[1] == "selftest" else run(sys.argv[2])

In [ ]:
%%writefile /kaggle/working/src/s23_ceband.py
"""Run X: fold-wise cross-encoder on the UNCERTAIN pairs of the final model (0.5 <= p < 0.995), as a stacker feature (s22).
A perfect judge on these pairs would lift labelled OOF 0.98685 -> 0.99138; the CE (reads both records' raw text, multilingual,
incl. French) tries to get part of that. Reuses s8_ce (training loop / scoring) with other pair sources and output names:
  train pairs : run R's OOF p7 (oof_p7_ge05.parquet) in the band, fold f = S1 buckets FOLDS[f], labels from train_gt
  OOF scores  : fold f pairs scored by the model trained on the other fold; test: pairs with p_m0 or p_m1 in the band, mean of both
  out: ceband_{split}_{j}.parquet [s1, id, ce]
usage: s23_ceband.py train <f> | score"""
import sys

import polars as pl

from common import WORK
import s8_ce

LO, HI = 0.5, 0.995


def band(split, j):
    if split == "train":
        return (pl.scan_parquet(WORK / "oof_p7_ge05.parquet").filter((pl.col("s1") % 8 == j) & pl.col("p").is_between(LO, HI, closed="left"))
                .select("s1", "id"))
    return (pl.scan_parquet(WORK / f"p7_v3_test_{j}.parquet")
            .filter(pl.col("p_m0").is_between(LO, HI, closed="left") | pl.col("p_m1").is_between(LO, HI, closed="left")).select("s1", "id"))


def ceband_path(split, j):
    return WORK / f"ceband_{split}_{j}.parquet"


s8_ce.uncertain = band            # the pairs s8_ce trains on / scores
s8_ce.ce_path = ceband_path       # its outputs (never the dataset's ce_v3_* files)
s8_ce.fold_dir = lambda f: WORK / f"ceband_model_f{f}"

if __name__ == "__main__":
    if sys.argv[1] == "train":
        s8_ce.train_fold(int(sys.argv[2]))
    elif sys.argv[1] == "score":
        s8_ce.predict_all()

In [ ]:
run = '#!/bin/bash\n# Run C: + candidates from the fine-tuned e5 (s17), then full-data stage 5 (+ a second, differently-configured stage-5 model and their blend) on Kaggle (31 GB RAM, 4 CPUs, T4).\nset -e\nW=/tmp/work; OUT=/kaggle/working; mkdir -p $W $OUT/state\nCORE=$(dirname "$(find /kaggle/input -name train_records.parquet | head -1)")\nEMB=$(dirname "$(find /kaggle/input -name train_emb.f16 | head -1)")\nV3=$(dirname "$(find /kaggle/input -name ce_v3_train_0.parquet | head -1)")\necho "inputs: core=$CORE emb=$EMB v3=$V3"\nfor d in "$CORE" "$EMB" "$V3"; do for f in "$d"/*; do case "$f" in *addr4_*) continue;; esac; [ -f "$f" ] && ln -sf "$f" $W/; done; done\nfor m in ce_model_f0 ce_model_f1; do d=$(find /kaggle/input -type d -name $m | head -1); [ -n "$d" ] && ln -sfn "$d" $W/$m || true; done   # not needed by stage 5/6\necho "linked $(ls $W | wc -l) inputs"\nfor f in train_records.parquet test_records.parquet train_gt.parquet train_norm2.parquet test_norm3.parquet train_feat2_0.parquet \\\n         train_emb.f16 test_emb.f16 knn_test_7.parquet ce_v3_test_7.parquet test_source1.tsv validate_submission.py; do\n  [ -e $W/$f ] || { echo "MISSING $f"; exit 1; }; done\ncd $OUT/src\nexport BER_WORK=$W BER_OUT=$OUT BER_FEAT=feat3 BER_TAG=_v3 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True BER_NPIECE=8\nfree -g | head -2\nVAL="python $W/validate_submission.py --test-dir $CORE --matching"\nFT=$(dirname "$(find /kaggle/input -name model.safetensors -path \'*ftmodel*\' | head -1)")\necho "fine-tuned e5: $FT"\nfor f in $(find /kaggle/input -name \'knnft_*.parquet\'); do ln -sf "$f" $W/; done\n[ -e "$FT/model.safetensors" ] || [ -e $W/knnft_test_7.parquet ] || { echo "MISSING fine-tuned model: add the mlc26-ftmodel dataset"; exit 1; }\nif [ ! -e $W/knnft_test_7.parquet ]; then\n  # the model was saved by transformers 5.x on Colab: prove it loads here (and is not randomly initialised) before 6 h of work\n  CHECK="from transformers import AutoModel, AutoTokenizer; import torch, transformers\nt = AutoTokenizer.from_pretrained(\'/tmp/e5ft\'); m = AutoModel.from_pretrained(\'/tmp/e5ft\').eval()\nx = t([\'query: sunrise pizza | 12 main street springfield\', \'query: sunrise pizza llc | 12 main st springfield\',\n       \'query: delhi dental clinic | 44 mg road pune\'], padding=True, return_tensors=\'pt\')\nwith torch.no_grad(): h = m(**x).last_hidden_state\ne = torch.nn.functional.normalize((h * x[\'attention_mask\'][..., None]).sum(1) / x[\'attention_mask\'].sum(1, keepdim=True), dim=1)\nsame, diff = float(e[0] @ e[1]), float(e[0] @ e[2])\nprint(f\'fine-tuned e5 loads (transformers {transformers.__version__}): cos same-business {same:.3f}, different {diff:.3f}\')\nassert same > 0.8 and same - diff > 0.05, \'fine-tuned model looks broken\'"\n  rm -rf /tmp/e5ft; mkdir -p /tmp/e5ft; cp -r $FT/* /tmp/e5ft/\n  if ! python -c "$CHECK"; then\n    ORIG=$(dirname "$(find /kaggle/input -name model.safetensors -path \'*multilingual-e5-small*\' | grep -v ftmodel | head -1)")\n    echo "fine-tuned dir did not load: using config + tokenizer of the original e5 ($ORIG) with the fine-tuned weights"\n    rm -rf /tmp/e5ft; mkdir -p /tmp/e5ft; cp -r $ORIG/* /tmp/e5ft/; cp $FT/model.safetensors /tmp/e5ft/model.safetensors\n    python -c "$CHECK"\n  fi\n  FT=/tmp/e5ft\n  echo "== encode train + test with the fine-tuned e5, one per GPU $(date)"\n  F=/tmp/ft; mkdir -p $F; ln -sf $W/train_records.parquet $W/test_records.parquet $F/\n  BER_WORK=$F BER_EMB_MODEL=$FT CUDA_VISIBLE_DEVICES=0 python e1_emb.py encode train > $OUT/ft_enc_train.log 2>&1 & P1=$!\n  BER_WORK=$F BER_EMB_MODEL=$FT CUDA_VISIBLE_DEVICES=1 python e1_emb.py encode test > $OUT/ft_enc_test.log 2>&1 & P2=$!\n  wait $P1; wait $P2; tail -n 2 $OUT/ft_enc_train.log $OUT/ft_enc_test.log; rm -f $F/*_emb_c*.f16\n  echo "== kNN@20 with the fine-tuned e5 $(date)"\n  BER_WORK=$F CUDA_VISIBLE_DEVICES=0 python e1_emb.py knn train > $OUT/ft_knn_train.log 2>&1 & P1=$!\n  BER_WORK=$F CUDA_VISIBLE_DEVICES=1 python e1_emb.py knn test > $OUT/ft_knn_test.log 2>&1 & P2=$!\n  wait $P1; wait $P2\n  mkdir -p $OUT/state/knnft\n  for s in train test; do for j in 0 1 2 3 4 5 6 7; do\n    cp $F/knn_${s}_${j}.parquet $OUT/state/knnft/knnft_${s}_${j}.parquet; ln -sf $OUT/state/knnft/knnft_${s}_${j}.parquet $W/; done; done\n  rm -rf $F\nfi\necho "== cosine $(date)"; python s10_v3.py cos train; python s10_v3.py cos test\necho "== feat3 $(date)"\nfor s in train test; do for j in 0 1 2 3 4 5 6 7; do python s10_v3.py build $s $j; done; done\npython s10_v3.py report\necho "== address add-on for the new union $(date)"\npython s14_addr.py train; python s14_addr.py test\necho "== stage5 FULL DATA $(date)"\npython s4_train.py 5 || { echo "== stage5 full data failed (likely RAM): retrying the unfinished fold(s) on half data"; BER_TRAIN_BUCKETS=2 python s4_train.py 5; }\ncp $W/val_stage5_v3.json $W/xgb_s5_v3_* $OUT/state/\npython s15_blend.py v3s5ft 5\n$VAL $OUT/v3s5ft/matching_results.tsv | tail -1; $VAL $OUT/v3s5ft_fr90/matching_results.tsv | tail -1\necho "== SUBMISSION READY: v3s5ft + v3s5ft_fr90 $(date)"\necho "== stage6: second stage-5 model (511 leaves, other seed/columns) $(date)"\nexport BER_LEAVES=511 BER_SEED=1 BER_COLS=0.7 BER_MCW=40\npython s4_train.py 6 || { echo "== stage6 full data failed: retrying on half data"; BER_TRAIN_BUCKETS=2 python s4_train.py 6; }\ncp $W/val_stage6_v3.json $W/xgb_s6_v3_* $OUT/state/\npython s15_blend.py blend56ft 5 6\n$VAL $OUT/blend56ft/matching_results.tsv | tail -1; $VAL $OUT/blend56ft_fr90/matching_results.tsv | tail -1\necho "== SUBMISSION READY: blend56ft + blend56ft_fr90 $(date)"\nmkdir -p $OUT/state/preds; cp $W/p5_v3_test_*.parquet $W/p6_v3_test_*.parquet $W/val_blend_*.json $OUT/state/preds/ 2>/dev/null || true\ndu -sh $OUT; echo "== KAGGLE RUN C DONE $(date)"\n'
open("/kaggle/working/run_kaggle.sh", "w").write(run)
print(subprocess.run("bash -n /kaggle/working/run_kaggle.sh && echo syntax-ok", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# runs the whole chain; output is streamed to the notebook and to /kaggle/working/log_kaggle.txt
p = subprocess.Popen("bash /kaggle/working/run_kaggle.sh 2>&1 | tee /kaggle/working/log_kaggle.txt", shell=True)
p.wait()
print("exit code", p.returncode)